# Trendyol term-grouped 5-fold OOF validation

This notebook is self-contained: it writes all required repository modules into
`/kaggle/working/src`. It builds candidate slates, trains leakage-safe
CatBoost/LightGBM folds, optionally trains transformer folds, combines OOF scores,
and optimizes Macro-F1 blend/rate/count constraints.

Recommended Kaggle workflow:

1. For a one-session attempt, use `STAGE = "all"` and all five folds.
2. If the session is too short, first run `STAGE = "build"`, save
   `/kaggle/working/grouped_oof` as a Kaggle dataset, attach it next session, set
   `CACHE_ROOT`, and run one or two folds at a time with `STAGE = "folds"`.
3. After all fold artifacts are cached, use `STAGE = "optimize"`.

For leakage-safe transformer OOF training, `BASE_MODEL` must be the original
pretrained encoder, not your previous full-data fine-tuned `bi-encoder` checkpoint.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# --------------------------- EDIT THIS CELL ---------------------------
STAGE = "all"  # "build", "folds", "optimize", or "all"
FOLDS_TO_RUN = [0, 1, 2, 3, 4]

RUN_CATBOOST = True
RUN_LGBM = True
RUN_TRANSFORMER = True

# Set this to an attached Kaggle dataset directory containing a previous
# grouped_oof output folder, for example:
# CACHE_ROOT = Path("/kaggle/input/my-trendyol-oof-cache/grouped_oof")
CACHE_ROOT = None

SKIP_EXISTING = True
N_SPLITS = 5
SEED = 42
BASE_CANDIDATES = 100

# Use the original pretrained model. With Internet disabled, attach an offline
# Kaggle dataset containing this Hugging Face model and set BASE_MODEL to it.
BASE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TRANSFORMER_EPOCHS = 2
TRANSFORMER_BATCH_SIZE = 64
TRANSFORMER_GRAD_ACCUM = 1

CATBOOST_ITERATIONS = 2000
CATBOOST_DEVICES = "0:1"  # CatBoost multi-GPU syntax uses a colon.
LGBM_ESTIMATORS = 4000
INNER_VALID_SIZE = 0.10
WEIGHT_STEP = 0.10

# "cuda" makes transformer_biencoder.py discover and use all visible GPUs.
# To select explicitly, use "cuda:0,1".
TRANSFORMER_DEVICE = "cuda"

PACKAGE_CACHE = True
# ---------------------------------------------------------------------

WORK_DIR = Path("/kaggle/working")
SRC_DIR = WORK_DIR / "src"
OUT_DIR = WORK_DIR / "grouped_oof"
SRC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

data_candidates = [
    Path("/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle"),
    Path("/kaggle/input/trendyol-e-ticaret-yarismasi-2026-kaggle"),
]
DATA_DIR = next(
    (path for path in data_candidates if (path / "training_pairs.csv").exists()), None
)
if DATA_DIR is None:
    matches = list(Path("/kaggle/input").glob("**/training_pairs.csv"))
    if len(matches) == 1:
        DATA_DIR = matches[0].parent
    else:
        raise FileNotFoundError(
            f"Could not uniquely locate competition data; matches={matches[:10]}"
        )

if CACHE_ROOT is not None:
    CACHE_ROOT = Path(CACHE_ROOT)
    if not CACHE_ROOT.exists():
        raise FileNotFoundError(CACHE_ROOT)
    shutil.copytree(CACHE_ROOT, OUT_DIR, dirs_exist_ok=True)
    print(f"restored cache from {CACHE_ROOT}")

print(f"STAGE={STAGE} FOLDS_TO_RUN={FOLDS_TO_RUN}")
print(f"DATA_DIR={DATA_DIR}")
print(f"OUT_DIR={OUT_DIR}")

## Install/check runtime dependencies

In [ ]:
import importlib.util

required_packages = {
    "rapidfuzz": "rapidfuzz",
    "text_unidecode": "text-unidecode",
    "catboost": "catboost",
    "lightgbm": "lightgbm",
    "sklearn": "scikit-learn",
}
if RUN_TRANSFORMER:
    required_packages.update({"torch": "torch", "transformers": "transformers"})

missing_packages = [
    pip_name
    for module, pip_name in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    print("installing", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True
    )
else:
    print("all dependencies are already available")

## Write embedded source modules

In [ ]:
%%writefile src/lexical_baseline.py
"""Build Turkish-aware lexical relevance features, scores, and submissions."""

from __future__ import annotations

import argparse
import csv
import math
import re
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz
from text_unidecode import unidecode
from tqdm import tqdm

TOKEN_RE = re.compile(r"[a-z0-9]+")

STOPWORDS = {
    "a",
    "ve",
    "ile",
    "icin",
    "bir",
    "adet",
    "ad",
    "li",
    "set",
    "seti",
    "takim",
    "takimi",
    "urun",
    "model",
    "uyumlu",
    "orjinal",
    "orijinal",
}

COLORS = {
    "beyaz",
    "siyah",
    "kirmizi",
    "mavi",
    "yesil",
    "sari",
    "pembe",
    "mor",
    "gri",
    "lacivert",
    "kahverengi",
    "bej",
    "krem",
    "turuncu",
    "bordo",
    "gold",
    "altin",
    "gumus",
    "haki",
    "taba",
    "lila",
    "ekru",
    "vizon",
    "füme",
}

GENDER_ALLOWED = {
    "kadin": {"kadin", "unisex", "unknown"},
    "bayan": {"kadin", "unisex", "unknown"},
    "erkek": {"erkek", "unisex", "unknown"},
    "unisex": {"kadin", "erkek", "unisex", "unknown"},
}

AGE_ALLOWED = {
    "bebek": {"bebek", "unknown"},
    "cocuk": {"cocuk", "bebek", "unknown"},
    "kiz": {"cocuk", "bebek", "unknown"},
}


def normalize_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = unidecode(str(value).lower())
    text = text.replace("&", " ")
    return re.sub(r"\s+", " ", text).strip()


def stem_token(token: str) -> str:
    if len(token) <= 4:
        return token

    for suffix in (
        "larin",
        "lerin",
        "lari",
        "leri",
        "lara",
        "lere",
        "larda",
        "lerde",
        "ndan",
        "nden",
        "dan",
        "den",
        "nin",
        "nun",
        "ler",
        "lar",
        "lik",
        "luk",
        "cuk",
        "cik",
        "ci",
        "cu",
        "si",
        "su",
        "u",
        "i",
    ):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[: -len(suffix)]
    return token


@lru_cache(maxsize=1_000_000)
def token_set(text: str) -> frozenset[str]:
    out: set[str] = set()
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        out.add(token)
        stemmed = stem_token(token)
        if stemmed not in STOPWORDS:
            out.add(stemmed)
    return frozenset(out)


def query_tokens(text: str) -> tuple[str, ...]:
    seen: set[str] = set()
    out: list[str] = []
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        stemmed = stem_token(token)
        for candidate in (token, stemmed):
            if candidate and candidate not in STOPWORDS and candidate not in seen:
                seen.add(candidate)
                out.append(candidate)
                break
    return tuple(out)


def coverage(query: tuple[str, ...], field: frozenset[str]) -> float:
    if not query:
        return 0.0
    return sum(1 for token in query if token in field) / len(query)


def color_tokens(query: tuple[str, ...]) -> set[str]:
    return {token for token in query if token in COLORS}


def load_terms(path: Path) -> dict[str, dict[str, object]]:
    terms = pd.read_csv(path)
    records: dict[str, dict[str, object]] = {}
    for row in terms.itertuples(index=False):
        norm_query = normalize_text(row.query)
        q_tokens = query_tokens(norm_query)
        records[row.term_id] = {
            "query": norm_query,
            "tokens": q_tokens,
            "colors": color_tokens(q_tokens),
        }
    return records


def load_items(path: Path) -> dict[str, tuple[str, str, str, str, str, str]]:
    items: dict[str, tuple[str, str, str, str, str, str]] = {}
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in tqdm(reader, desc="items"):
            item_id = row["item_id"]
            items[item_id] = (
                normalize_text(row.get("title")),
                normalize_text(row.get("category")),
                normalize_text(row.get("brand")),
                normalize_text(row.get("gender")) or "unknown",
                normalize_text(row.get("age_group")) or "unknown",
                normalize_text(row.get("attributes")),
            )
    return items


def pair_features(
    term: dict[str, object], item: tuple[str, str, str, str, str, str]
) -> dict[str, float]:
    query = term["query"]  # type: ignore[index]
    q_tokens = term["tokens"]  # type: ignore[index]
    q_colors = term["colors"]  # type: ignore[index]
    title, category, brand, gender, age_group, attrs = item

    title_tokens = token_set(title)
    category_tokens = token_set(category)
    brand_tokens = token_set(brand)
    attr_tokens = token_set(attrs)
    full_tokens = title_tokens | category_tokens | brand_tokens | attr_tokens

    title_cov = coverage(q_tokens, title_tokens)
    category_cov = coverage(q_tokens, category_tokens)
    brand_cov = coverage(q_tokens, brand_tokens)
    attr_cov = coverage(q_tokens, attr_tokens)
    full_cov = coverage(q_tokens, full_tokens)

    title_ratio = fuzz.token_set_ratio(query, title) / 100.0
    category_ratio = fuzz.token_set_ratio(query, category) / 100.0
    brand_ratio = max(
        fuzz.ratio(query, brand) / 100.0,
        fuzz.partial_ratio(query, brand) / 100.0 if brand else 0.0,
    )

    query_in_title = float(bool(query and query in title))
    query_in_category = float(bool(query and query in category))
    query_in_brand = float(bool(query and query in brand))
    full_complete = float(full_cov >= 0.999)
    full_high = float(0.75 <= full_cov < 0.999)
    title_complete = float(title_cov >= 0.999)
    category_complete = float(category_cov >= 0.999)
    brand_complete = float(brand_cov >= 0.999 and bool(brand))
    short_complete = float(len(q_tokens) <= 2 and full_cov >= 0.999)

    gender_mismatch = 0.0
    for gender_word, allowed in GENDER_ALLOWED.items():
        if (
            gender_word in q_tokens
            and gender not in allowed
            and gender_word not in full_tokens
        ):
            gender_mismatch = 1.0
            break

    age_mismatch = 0.0
    for age_word, allowed in AGE_ALLOWED.items():
        if (
            age_word in q_tokens
            and age_group not in allowed
            and age_word not in full_tokens
        ):
            age_mismatch = 1.0
            break

    color_mismatch = float(bool(q_colors and full_tokens.isdisjoint(q_colors)))
    weak_match = float(
        full_cov < 0.34 and max(title_ratio, category_ratio, brand_ratio) < 0.55
    )

    features = {
        "n_query_tokens": float(len(q_tokens)),
        "title_cov": title_cov,
        "category_cov": category_cov,
        "brand_cov": brand_cov,
        "attr_cov": attr_cov,
        "full_cov": full_cov,
        "title_ratio": title_ratio,
        "category_ratio": category_ratio,
        "brand_ratio": brand_ratio,
        "query_in_title": query_in_title,
        "query_in_category": query_in_category,
        "query_in_brand": query_in_brand,
        "full_complete": full_complete,
        "full_high": full_high,
        "title_complete": title_complete,
        "category_complete": category_complete,
        "brand_complete": brand_complete,
        "short_complete": short_complete,
        "gender_mismatch": gender_mismatch,
        "age_mismatch": age_mismatch,
        "color_mismatch": color_mismatch,
        "weak_match": weak_match,
        "title_len": float(len(title_tokens)),
        "category_len": float(len(category_tokens)),
        "brand_len": float(len(brand_tokens)),
        "attr_len": float(len(attr_tokens)),
    }

    features["lexical_score"] = lexical_score_from_features(features)
    return features


def lexical_score_from_features(features: dict[str, float]) -> float:
    title_cov = features["title_cov"]
    category_cov = features["category_cov"]
    brand_cov = features["brand_cov"]
    attr_cov = features["attr_cov"]
    full_cov = features["full_cov"]
    title_ratio = features["title_ratio"]
    category_ratio = features["category_ratio"]
    brand_ratio = features["brand_ratio"]

    score = 0.0
    score += 2.1 * full_cov
    score += 1.4 * title_cov
    score += 1.0 * category_cov
    score += 1.2 * brand_cov
    score += 0.55 * attr_cov
    score += 0.75 * title_ratio
    score += 0.45 * category_ratio
    score += 0.55 * brand_ratio

    if features["query_in_title"]:
        score += 1.0
    if features["query_in_category"]:
        score += 0.8
    if features["query_in_brand"]:
        score += 1.1
    if features["full_complete"]:
        score += 1.2
    elif features["full_high"]:
        score += 0.55
    if features["title_complete"]:
        score += 0.9
    if features["category_complete"]:
        score += 0.7
    if features["brand_complete"]:
        score += 0.9
    if features["short_complete"]:
        score += 0.55

    if features["gender_mismatch"]:
        score -= 0.85
    if features["age_mismatch"]:
        score -= 0.75
    if features["color_mismatch"]:
        score -= 0.7
    if features["weak_match"]:
        score -= 0.9

    return score


def score_pair(
    term: dict[str, object], item: tuple[str, str, str, str, str, str]
) -> float:
    return pair_features(term, item)["lexical_score"]


def score_pairs(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")

    in_path = data_dir / args.pairs
    total = args.limit
    if total is None:
        with in_path.open(newline="", encoding="utf-8") as handle:
            total = sum(1 for _ in handle) - 1

    with in_path.open(newline="", encoding="utf-8") as in_handle, out_path.open(
        "w", newline="", encoding="utf-8"
    ) as out_handle:
        reader = csv.DictReader(in_handle)
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "score"])
        for i, row in enumerate(tqdm(reader, total=total, desc="pairs")):
            if args.limit is not None and i >= args.limit:
                break
            item = items[row["item_id"]]
            term = terms[row["term_id"]]
            writer.writerow(
                [row["id"], row["term_id"], f"{score_pair(term, item):.6f}"]
            )


def summarize_scores(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    print(
        scores["score"].describe(
            percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
        )
    )
    for threshold in args.thresholds:
        pred = scores["score"] >= threshold
        print(
            f"threshold={threshold:.3f} positives={int(pred.sum())} rate={pred.mean():.5f}"
        )

    by_term = scores.groupby("term_id")["score"].agg(["count", "max", "mean"])
    print(by_term.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


def make_submission(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    pred = scores["score"] >= args.threshold

    if args.min_per_term > 0 or args.max_per_term > 0:
        scores = scores.assign(pred=pred.astype("int8"))
        parts = []
        for _, group in tqdm(scores.groupby("term_id", sort=False), desc="terms"):
            group = group.copy()
            order = group["score"].sort_values(ascending=False).index
            if (
                args.min_per_term > 0
                and group.loc[order[: args.min_per_term], "score"].max()
                >= args.min_score
            ):
                group.loc[order[: args.min_per_term], "pred"] = 1
            if args.max_per_term > 0:
                positives = group.index[group["pred"] == 1]
                if len(positives) > args.max_per_term:
                    keep = (
                        group.loc[positives]
                        .sort_values("score", ascending=False)
                        .head(args.max_per_term)
                        .index
                    )
                    group.loc[positives, "pred"] = 0
                    group.loc[keep, "pred"] = 1
            parts.append(group[["id", "pred"]])
        scored = pd.concat(parts, ignore_index=True)
        pred_values = scored["pred"].astype("int8")
        ids = scored["id"]
    else:
        pred_values = pred.astype("int8")
        ids = scores["id"]

    output = pd.DataFrame({"id": ids, "prediction": pred_values})
    output.to_csv(args.output, index=False)
    counts = Counter(output["prediction"])
    print(f"wrote {args.output}")
    print(
        f"rows={len(output)} positives={counts[1]} rate={counts[1] / len(output):.5f}"
    )


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    score = sub.add_parser("score")
    score.add_argument("--data-dir", default="data")
    score.add_argument("--pairs", default="submission_pairs.csv")
    score.add_argument("--output", default="outputs/lexical_scores.csv")
    score.add_argument("--limit", type=int)
    score.set_defaults(func=score_pairs)

    summary = sub.add_parser("summary")
    summary.add_argument("--scores", default="outputs/lexical_scores.csv")
    summary.add_argument(
        "--thresholds", type=float, nargs="+", default=[3.0, 3.5, 4.0, 4.5, 5.0]
    )
    summary.set_defaults(func=summarize_scores)

    submit = sub.add_parser("submit")
    submit.add_argument("--scores", default="outputs/lexical_scores.csv")
    submit.add_argument("--output", default="outputs/submission_lexical.csv")
    submit.add_argument("--threshold", type=float, required=True)
    submit.add_argument("--min-per-term", type=int, default=0)
    submit.add_argument("--max-per-term", type=int, default=0)
    submit.add_argument("--min-score", type=float, default=0.0)
    submit.set_defaults(func=make_submission)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_term_negatives.py
"""Mine deterministic TF-IDF negatives from training queries and catalog items."""

from __future__ import annotations

import argparse
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

from lexical_baseline import normalize_text


def item_search_text(items: pd.DataFrame) -> pd.Series:
    title = items["title"].map(normalize_text)
    category = items["category"].map(normalize_text)
    brand = items["brand"].map(normalize_text)
    attrs = items["attributes"].map(normalize_text)
    return (
        title + " " + title + " " + brand + " " + brand + " " + category + " " + attrs
    ).str.strip()


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def topk_sparse_row(row: sparse.spmatrix, k: int) -> tuple[np.ndarray, np.ndarray]:
    row = row.tocsr()
    indices = row.indices
    data = row.data
    if len(data) == 0:
        return indices, data

    if len(data) > k:
        selected = np.argpartition(data, -k)[-k:]
        selected = selected[np.argsort(-data[selected], kind="mergesort")]
    else:
        selected = np.argsort(-data, kind="mergesort")
    return indices[selected], data[selected]


def sample_easy_items(
    item_count: int,
    excluded: set[int],
    n_items: int,
    rng: np.random.Generator,
) -> list[int]:
    if n_items <= 0:
        return []

    selected: list[int] = []
    attempts = 0
    max_attempts = max(100, n_items * 50)
    while len(selected) < n_items and attempts < max_attempts:
        item_pos = int(rng.integers(0, item_count))
        attempts += 1
        if item_pos in excluded:
            continue
        excluded.add(item_pos)
        selected.append(item_pos)
    return selected


def load_positive_pairs(path: Path) -> pd.DataFrame:
    pairs = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "label" in pairs.columns:
        pairs = pairs.loc[pairs["label"].astype(str).ne("0")]
    pairs = pairs[["term_id", "item_id"]].drop_duplicates().reset_index(drop=True)
    if pairs.empty:
        raise ValueError(f"No positive training pairs found in {path}")
    return pairs


def mine_negatives(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    terms = pd.read_csv(data_dir / "terms.csv", dtype=str, keep_default_na=False)
    items = pd.read_csv(
        data_dir / "items.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["item_id", "title", "category", "brand", "attributes"],
    )
    positives = load_positive_pairs(data_dir / "training_pairs.csv")

    train_term_ids = positives["term_id"].drop_duplicates().to_numpy()
    if args.limit_terms:
        train_term_ids = train_term_ids[: args.limit_terms]
        positives = positives.loc[
            positives["term_id"].isin(train_term_ids)
        ].reset_index(drop=True)
    if args.limit_items:
        items = items.head(args.limit_items).reset_index(drop=True)

    terms = terms.loc[
        terms["term_id"].isin(train_term_ids), ["term_id", "query"]
    ].copy()
    missing_terms = set(train_term_ids) - set(terms["term_id"])
    if missing_terms:
        examples = sorted(missing_terms)[:5]
        raise KeyError(
            f"{len(missing_terms)} training term_ids are missing from terms.csv, "
            f"examples={examples}"
        )

    terms["query_norm"] = terms["query"].map(normalize_text)
    terms = terms.set_index("term_id").loc[train_term_ids].reset_index()
    item_ids = items["item_id"].astype(str).to_numpy()
    item_to_pos = {item_id: i for i, item_id in enumerate(item_ids)}

    positive_by_term: dict[str, set[int]] = {}
    for term_id, group in positives.groupby("term_id", sort=False):
        positions = {
            item_to_pos[item_id]
            for item_id in group["item_id"].astype(str)
            if item_id in item_to_pos
        }
        positive_by_term[str(term_id)] = positions

    item_text = item_search_text(items)
    corpus = pd.concat([terms["query_norm"], item_text], ignore_index=True)
    vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, args.word_ngram_max),
        min_df=args.min_df,
        max_features=args.max_features,
        dtype=np.float32,
        norm="l2",
        sublinear_tf=True,
    )
    vectorizer.fit(corpus)
    term_matrix = vectorizer.transform(terms["query_norm"])
    item_matrix = vectorizer.transform(item_text)
    item_matrix_t = item_matrix.T.tocsr()
    print(
        f"terms={len(terms):,} items={len(items):,} positives={len(positives):,} "
        f"vocab={len(vectorizer.vocabulary_):,}"
    )

    rng = np.random.default_rng(args.seed)
    counts = {"hard": 0, "mid": 0, "easy": 0}
    total = 0
    positive_keys = set(pair_key(positives))

    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["term_id", "item_id", "negative_band", "tfidf_score"])

        for start in tqdm(
            range(0, len(terms), args.chunk_size), desc="mine train-term negatives"
        ):
            stop = min(start + args.chunk_size, len(terms))
            scores = term_matrix[start:stop] @ item_matrix_t
            scores = scores.tocsr()

            for row_offset, term_id in enumerate(
                terms["term_id"].iloc[start:stop].astype(str)
            ):
                pos_items = set(positive_by_term.get(term_id, set()))
                excluded = set(pos_items)
                row_indices, row_scores = topk_sparse_row(
                    scores.getrow(row_offset), args.retrieve_topk
                )

                candidates: list[tuple[int, float]] = []
                for item_pos, score in zip(row_indices, row_scores):
                    item_pos = int(item_pos)
                    if item_pos in excluded:
                        continue
                    excluded.add(item_pos)
                    candidates.append((item_pos, float(score)))

                hard = candidates[: args.hard_per_term]
                mid_pool = candidates[args.hard_per_term :]
                if len(mid_pool) > args.mid_per_term:
                    chosen = rng.choice(
                        len(mid_pool), size=args.mid_per_term, replace=False
                    )
                    mid = [mid_pool[int(i)] for i in chosen]
                else:
                    mid = mid_pool

                rows: list[tuple[str, str, str, float]] = []
                for item_pos, score in hard:
                    rows.append((term_id, item_ids[item_pos], "hard", score))
                for item_pos, score in mid:
                    rows.append((term_id, item_ids[item_pos], "mid", score))

                easy_positions = sample_easy_items(
                    len(item_ids), excluded, args.easy_per_term, rng
                )
                for item_pos in easy_positions:
                    rows.append((term_id, item_ids[item_pos], "easy", 0.0))

                for out_term_id, item_id, band, score in rows:
                    if f"{out_term_id}\t{item_id}" in positive_keys:
                        continue
                    writer.writerow([out_term_id, item_id, band, f"{score:.8f}"])
                    counts[band] += 1
                    total += 1

    print(f"wrote {output_path}")
    print(
        f"negatives={total:,} hard={counts['hard']:,} "
        f"mid={counts['mid']:,} easy={counts['easy']:,}"
    )


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--output", default="outputs/train_term_negatives.csv")
    parser.add_argument("--hard-per-term", type=int, default=25)
    parser.add_argument("--mid-per-term", type=int, default=15)
    parser.add_argument("--easy-per-term", type=int, default=10)
    parser.add_argument("--retrieve-topk", type=int, default=300)
    parser.add_argument("--chunk-size", type=int, default=128)
    parser.add_argument("--max-features", type=int, default=250_000)
    parser.add_argument("--min-df", type=int, default=2)
    parser.add_argument("--word-ngram-max", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--limit-terms",
        type=int,
        help="Debug only: mine negatives for the first N train terms.",
    )
    parser.add_argument(
        "--limit-items", type=int, help="Debug only: restrict the item catalog."
    )
    args = parser.parse_args()
    mine_negatives(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pu_catboost.py
"""Train and score a category-aware positive-unlabeled CatBoost model."""

from __future__ import annotations

import argparse
import csv
import gc
import math
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from lexical_baseline import load_items, load_terms, normalize_text, pair_features

BASE_FEATURE_NAMES = [
    "n_query_tokens",
    "title_cov",
    "category_cov",
    "brand_cov",
    "attr_cov",
    "full_cov",
    "title_ratio",
    "category_ratio",
    "brand_ratio",
    "query_in_title",
    "query_in_category",
    "query_in_brand",
    "full_complete",
    "full_high",
    "title_complete",
    "category_complete",
    "brand_complete",
    "short_complete",
    "gender_mismatch",
    "age_mismatch",
    "color_mismatch",
    "weak_match",
    "title_len",
    "category_len",
    "brand_len",
    "attr_len",
    "lexical_score",
]

CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5
CATEGORY_FEATURE_NAMES: list[str] = []
for level in CATEGORY_LEVELS:
    CATEGORY_FEATURE_NAMES.extend(
        [
            f"{level}_top1_match",
            f"{level}_top3_match",
            f"{level}_top5_match",
            f"{level}_candidate_prob",
            f"{level}_top1_prob",
            f"{level}_rank",
        ]
    )

ITEM_CAT_FEATURES = ["item_cat_l1", "item_cat_l2", "item_gender", "item_age_group"]
FEATURE_NAMES = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES + ITEM_CAT_FEATURES
CAT_FEATURES = ITEM_CAT_FEATURES


def split_category(value: object) -> tuple[str, str, str, str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        parts = ["unknown"]
    else:
        parts = [normalize_text(part) for part in str(value).split("/")]
        parts = [part for part in parts if part]
        if not parts:
            parts = ["unknown"]

    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf


def add_category_levels(
    frame: pd.DataFrame, category_col: str = "category"
) -> pd.DataFrame:
    levels = pd.DataFrame(
        frame[category_col].map(split_category).tolist(),
        columns=["cat_l1", "cat_l2", "cat_l3", "cat_leaf"],
        index=frame.index,
    )
    return pd.concat([frame, levels], axis=1)


def build_term_category_lookup(
    topk: pd.DataFrame,
    levels: tuple[str, ...] = CATEGORY_LEVELS,
    k: int = CATEGORY_TOPK,
) -> dict[str, dict[str, object]]:
    lookup: dict[str, dict[str, object]] = {}
    for row in topk.itertuples(index=False):
        row_dict = row._asdict()
        entry: dict[str, object] = {}
        for level in levels:
            scores: dict[str, float] = {}
            for rank in range(1, k + 1):
                label = row_dict.get(f"{level}_top{rank}")
                score = row_dict.get(f"{level}_prob{rank}")
                if isinstance(label, str) and label:
                    scores[label] = float(score)
            entry[level] = scores
            entry[f"{level}_top1_prob"] = next(iter(scores.values()), 0.0)
        lookup[row_dict["term_id"]] = entry
    return lookup


def build_item_meta_lookup(items_path: Path) -> dict[str, dict[str, str]]:
    items = pd.read_csv(
        items_path, usecols=["item_id", "category", "gender", "age_group"]
    )
    items = add_category_levels(items)
    lookup: dict[str, dict[str, str]] = {}
    for row in tqdm(
        items.itertuples(index=False), total=len(items), desc="item metadata"
    ):
        lookup[row.item_id] = {
            "cat_l1": row.cat_l1,
            "cat_l2": row.cat_l2,
            "item_cat_l1": row.cat_l1 or "unknown",
            "item_cat_l2": row.cat_l2 or "unknown",
            "item_gender": normalize_text(row.gender) or "unknown",
            "item_age_group": normalize_text(row.age_group) or "unknown",
        }
    return lookup


def category_pair_features(
    term_id: str,
    item_id: str,
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
) -> dict[str, float | str]:
    term_entry = term_category_lookup.get(term_id, {})
    item_entry = item_meta_lookup.get(item_id, {})
    out: dict[str, float | str] = {
        "item_cat_l1": item_entry.get("item_cat_l1", "unknown"),
        "item_cat_l2": item_entry.get("item_cat_l2", "unknown"),
        "item_gender": item_entry.get("item_gender", "unknown"),
        "item_age_group": item_entry.get("item_age_group", "unknown"),
    }

    for level in CATEGORY_LEVELS:
        scores = term_entry.get(level, {})
        if not isinstance(scores, dict):
            scores = {}
        ordered_labels = list(scores.keys())
        candidate = item_entry.get(level, "")
        rank = ordered_labels.index(candidate) + 1 if candidate in scores else 0

        out[f"{level}_top1_match"] = float(rank == 1)
        out[f"{level}_top3_match"] = float(1 <= rank <= 3)
        out[f"{level}_top5_match"] = float(1 <= rank <= 5)
        out[f"{level}_candidate_prob"] = float(scores.get(candidate, 0.0))
        out[f"{level}_top1_prob"] = float(term_entry.get(f"{level}_top1_prob", 0.0))
        out[f"{level}_rank"] = float(rank)

    return out


def rows_to_frame(
    rows: pd.DataFrame,
    terms: dict[str, dict[str, object]],
    items: dict[str, tuple[str, str, str, str, str, str]],
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
    desc: str,
) -> pd.DataFrame:
    numeric_names = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES
    numeric = np.empty((len(rows), len(numeric_names)), dtype=np.float32)
    cat_values = {name: np.empty(len(rows), dtype=object) for name in ITEM_CAT_FEATURES}

    iterator = tqdm(rows.itertuples(index=False), total=len(rows), desc=desc)
    for i, row in enumerate(iterator):
        features: dict[str, float | str] = pair_features(
            terms[row.term_id], items[row.item_id]
        )
        features.update(
            category_pair_features(
                row.term_id, row.item_id, term_category_lookup, item_meta_lookup
            )
        )
        numeric[i] = [float(features.get(name, 0.0)) for name in numeric_names]
        for name in ITEM_CAT_FEATURES:
            cat_values[name][i] = features.get(name, "unknown") or "unknown"

    frame = pd.DataFrame(numeric, columns=numeric_names)
    for name in ITEM_CAT_FEATURES:
        frame[name] = cat_values[name]
    return frame[FEATURE_NAMES]


def sample_frame(
    frame: pd.DataFrame, mask: pd.Series, n: int, seed: int
) -> pd.DataFrame:
    subset = frame.loc[mask]
    if len(subset) <= n:
        return subset
    return subset.sample(n=n, random_state=seed)


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def load_train_term_negatives(path: Path, positives: pd.DataFrame) -> pd.DataFrame:
    negatives = pd.read_csv(
        path, usecols=["term_id", "item_id"], dtype=str, keep_default_na=False
    )
    negatives = negatives.drop_duplicates(["term_id", "item_id"]).reset_index(drop=True)
    positive_keys = set(pair_key(positives))
    keep = ~pair_key(negatives).isin(positive_keys)
    dropped = int((~keep).sum())
    negatives = negatives.loc[keep].reset_index(drop=True)
    if negatives.empty:
        raise ValueError(f"No usable negatives found in {path}")
    if dropped:
        print(f"dropped {dropped:,} negatives that overlap known positives")
    print(f"using train-term negatives from {path} rows={len(negatives):,}")
    return negatives


def build_unlabeled_negatives(args: argparse.Namespace) -> pd.DataFrame:
    negative_path = Path(args.negatives) if args.negatives else None
    if negative_path and negative_path.exists():
        positives = pd.read_csv(
            Path(args.data_dir) / "training_pairs.csv",
            usecols=["term_id", "item_id"],
            dtype=str,
        )
        return load_train_term_negatives(negative_path, positives)

    if not args.allow_submission_negatives:
        raise FileNotFoundError(
            f"Train-term negatives were not found at {negative_path}. "
            "Create train-term negatives first, or pass "
            "`--allow-submission-negatives` to use the older submission-pair fallback."
        )

    data_dir = Path(args.data_dir)
    pairs = pd.read_csv(
        data_dir / "submission_pairs.csv", usecols=["id", "term_id", "item_id"]
    )
    scores = pd.read_csv(args.lexical_scores, usecols=["score"])
    if len(pairs) != len(scores):
        raise ValueError("submission_pairs and lexical_scores row counts differ")

    pairs["score"] = scores["score"].astype("float32").to_numpy()
    parts = [
        sample_frame(pairs, pairs["score"] < args.low_max, args.n_low, args.seed),
        sample_frame(
            pairs,
            (pairs["score"] >= args.low_max) & (pairs["score"] < args.mid_max),
            args.n_mid,
            args.seed + 1,
        ),
        sample_frame(
            pairs,
            (pairs["score"] >= args.mid_max) & (pairs["score"] < args.hard_max),
            args.n_hard,
            args.seed + 2,
        ),
    ]
    negatives = pd.concat(parts, ignore_index=True).drop_duplicates("id")
    print("negative score distribution")
    print(negatives["score"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).to_string())
    return negatives[["term_id", "item_id"]]


def load_feature_context(args: argparse.Namespace):
    data_dir = Path(args.data_dir)
    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")
    item_meta_lookup = build_item_meta_lookup(data_dir / "items.csv")

    topk_path = Path(args.term_category_topk)
    if not topk_path.exists():
        raise FileNotFoundError(
            f"term category top-k file not found: {topk_path}. "
            "Run the category-aware notebook first or pass --term-category-topk."
        )
    term_category_topk = pd.read_csv(topk_path)
    term_category_lookup = build_term_category_lookup(term_category_topk)
    return terms, items, term_category_lookup, item_meta_lookup


def train(args: argparse.Namespace) -> None:
    start_time = time.time()
    model_path = Path(args.model)
    model_path.parent.mkdir(parents=True, exist_ok=True)

    terms, items, term_category_lookup, item_meta_lookup = load_feature_context(args)
    positives = pd.read_csv(
        Path(args.data_dir) / "training_pairs.csv", usecols=["term_id", "item_id"]
    )
    negatives = build_unlabeled_negatives(args)

    if args.n_pos and len(positives) > args.n_pos:
        positives = positives.sample(n=args.n_pos, random_state=args.seed)

    x_pos = rows_to_frame(
        positives,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        "positive features",
    )
    x_neg = rows_to_frame(
        negatives,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        "negative features",
    )
    x = pd.concat([x_pos, x_neg], ignore_index=True)
    y = np.concatenate(
        [np.ones(len(x_pos), dtype=np.int8), np.zeros(len(x_neg), dtype=np.int8)]
    )
    print(f"matrix={x.shape} positives={int(y.sum()):,} positive_rate={y.mean():.4f}")

    del x_pos, x_neg
    gc.collect()

    x_train, x_valid, y_train, y_valid = train_test_split(
        x,
        y,
        test_size=args.valid_size,
        random_state=args.seed,
        stratify=y,
    )
    del x
    gc.collect()

    model_params = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "iterations": args.iterations,
        "learning_rate": args.learning_rate,
        "depth": args.depth,
        "l2_leaf_reg": args.l2_leaf_reg,
        "random_strength": args.random_strength,
        "bootstrap_type": "Bernoulli",
        "subsample": args.subsample,
        "auto_class_weights": "Balanced",
        "random_seed": args.seed,
        "thread_count": args.thread_count,
        "allow_writing_files": False,
        "verbose": args.verbose,
        "task_type": args.task_type,
        "devices": args.devices,
    }
    if args.task_type == "CPU":
        model_params["rsm"] = args.rsm
    elif args.rsm != 1.0:
        print("CatBoost GPU ignores rsm for Logloss; training without rsm.")

    model = CatBoostClassifier(**model_params)

    train_pool = Pool(x_train, y_train, cat_features=CAT_FEATURES)
    valid_pool = Pool(x_valid, y_valid, cat_features=CAT_FEATURES)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=False)

    valid_prob = model.predict_proba(valid_pool)[:, 1]
    best = (0.0, 0.5)
    for threshold in np.linspace(0.05, 0.95, 91):
        pred = (valid_prob >= threshold).astype(np.int8)
        score = f1_score(y_valid, pred, average="macro")
        if score > best[0]:
            best = (float(score), float(threshold))
    print(f"synthetic valid macro_f1={best[0]:.5f} threshold={best[1]:.3f}")

    importances = pd.Series(
        model.get_feature_importance(valid_pool), index=FEATURE_NAMES
    ).sort_values(ascending=False)
    print(importances.head(35).to_string())

    model.save_model(str(model_path))
    joblib.dump(
        {
            "features": FEATURE_NAMES,
            "cat_features": CAT_FEATURES,
            "threshold": best[1],
            "iterations": args.iterations,
            "term_category_topk": str(args.term_category_topk),
        },
        model_path.with_suffix(".meta.joblib"),
    )
    elapsed = time.time() - start_time
    print(f"wrote {model_path}")
    print(f"elapsed_minutes={elapsed / 60:.1f}")


def predict(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    model = CatBoostClassifier()
    model.load_model(str(args.model))
    terms, items, term_category_lookup, item_meta_lookup = load_feature_context(args)

    written = 0
    with output_path.open("w", newline="", encoding="utf-8") as out_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "prob"])

        reader = pd.read_csv(
            data_dir / "submission_pairs.csv", chunksize=args.chunk_size
        )
        for chunk in tqdm(reader, desc="predict chunks"):
            matrix = rows_to_frame(
                chunk,
                terms,
                items,
                term_category_lookup,
                item_meta_lookup,
                "chunk features",
            )
            pool = Pool(matrix, cat_features=CAT_FEATURES)
            prob = model.predict_proba(pool)[:, 1]
            writer.writerows(
                (row_id, term_id, f"{score:.8f}")
                for row_id, term_id, score in zip(chunk["id"], chunk["term_id"], prob)
            )
            written += len(chunk)
            del matrix, pool, prob, chunk
            gc.collect()

    print(f"wrote {output_path} rows={written:,}")


def make_submission(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    if args.rate is not None:
        threshold = scores["prob"].quantile(1.0 - args.rate)
    else:
        threshold = args.threshold
    pred = (scores["prob"] >= threshold).astype("int8")
    output = pd.DataFrame({"id": scores["id"], "prediction": pred})
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(out_path, index=False)
    print(f"threshold={threshold:.8f}")
    print(f"wrote {out_path}")
    print(f"rows={len(output):,} positives={int(pred.sum()):,} rate={pred.mean():.5f}")


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    common = argparse.ArgumentParser(add_help=False)
    common.add_argument("--data-dir", default="data")
    common.add_argument(
        "--term-category-topk",
        default="outputs/category_aware_notebook/term_category_topk.csv",
    )

    train_cmd = sub.add_parser("train", parents=[common])
    train_cmd.add_argument("--lexical-scores", default="outputs/lexical_scores.csv")
    train_cmd.add_argument("--negatives", default="outputs/train_term_negatives.csv")
    train_cmd.add_argument("--allow-submission-negatives", action="store_true")
    train_cmd.add_argument("--model", default="outputs/catboost_category_aware.cbm")
    train_cmd.add_argument("--n-pos", type=int, default=0)
    train_cmd.add_argument("--n-low", type=int, default=350_000)
    train_cmd.add_argument("--n-mid", type=int, default=350_000)
    train_cmd.add_argument("--n-hard", type=int, default=150_000)
    train_cmd.add_argument("--low-max", type=float, default=2.0)
    train_cmd.add_argument("--mid-max", type=float, default=4.5)
    train_cmd.add_argument("--hard-max", type=float, default=6.5)
    train_cmd.add_argument("--valid-size", type=float, default=0.15)
    train_cmd.add_argument("--iterations", type=int, default=5_000)
    train_cmd.add_argument("--learning-rate", type=float, default=0.025)
    train_cmd.add_argument("--depth", type=int, default=6)
    train_cmd.add_argument("--l2-leaf-reg", type=float, default=6.0)
    train_cmd.add_argument("--random-strength", type=float, default=0.5)
    train_cmd.add_argument("--subsample", type=float, default=0.85)
    train_cmd.add_argument("--rsm", type=float, default=0.95)
    train_cmd.add_argument("--thread-count", type=int, default=-1)
    train_cmd.add_argument("--verbose", type=int, default=100)
    train_cmd.add_argument("--task-type", choices=["CPU", "GPU"], default="CPU")
    train_cmd.add_argument("--devices", default="0")
    train_cmd.add_argument("--seed", type=int, default=42)
    train_cmd.set_defaults(func=train)

    pred_cmd = sub.add_parser("predict", parents=[common])
    pred_cmd.add_argument("--model", default="outputs/catboost_category_aware.cbm")
    pred_cmd.add_argument(
        "--output", default="outputs/catboost_category_aware_scores.csv"
    )
    pred_cmd.add_argument("--chunk-size", type=int, default=150_000)
    pred_cmd.set_defaults(func=predict)

    submit_cmd = sub.add_parser("submit")
    submit_cmd.add_argument(
        "--scores", default="outputs/catboost_category_aware_scores.csv"
    )
    submit_cmd.add_argument("--output", default="outputs/catboost_category_aware.csv")
    submit_cmd.add_argument("--threshold", type=float, default=0.5)
    submit_cmd.add_argument("--rate", type=float)
    submit_cmd.set_defaults(func=make_submission)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/category_signal.py
"""Learn query-to-category priors for category-aware relevance models."""

from __future__ import annotations

import math
import time
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from tqdm import tqdm

from lexical_baseline import normalize_text

CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5


def split_category(value: object) -> tuple[str, str, str, str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        parts = ["unknown"]
    else:
        parts = [normalize_text(part) for part in str(value).split("/")]
        parts = [part for part in parts if part]
        if not parts:
            parts = ["unknown"]

    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf


def add_category_levels(
    frame: pd.DataFrame, category_col: str = "category"
) -> pd.DataFrame:
    levels = pd.DataFrame(
        frame[category_col].map(split_category).tolist(),
        columns=["cat_l1", "cat_l2", "cat_l3", "cat_leaf"],
        index=frame.index,
    )
    return pd.concat([frame, levels], axis=1)


def build_category_examples(
    terms: pd.DataFrame,
    items: pd.DataFrame,
    positive_pairs: pd.DataFrame,
    max_examples: int | None = None,
    seed: int = 42,
) -> pd.DataFrame:
    required_term_ids = set(positive_pairs["term_id"].astype(str))
    required_item_ids = set(positive_pairs["item_id"].astype(str))
    term_subset = terms.loc[
        terms["term_id"].astype(str).isin(required_term_ids), ["term_id", "query"]
    ]
    item_subset = items.loc[
        items["item_id"].astype(str).isin(required_item_ids), ["item_id", "category"]
    ]

    examples = (
        positive_pairs[["term_id", "item_id"]]
        .merge(term_subset, on="term_id", how="left", validate="many_to_one")
        .merge(item_subset, on="item_id", how="left", validate="many_to_one")
        .dropna(subset=["query", "category"])
    )
    examples = add_category_levels(examples)
    examples["query_norm"] = examples["query"].map(normalize_text)

    # A broad query can have hundreds of positive products in the same category.
    # Keep one query/category path so it does not dominate the category classifier.
    examples = examples.drop_duplicates(["query_norm", "cat_l2"])
    if max_examples is not None and len(examples) > max_examples:
        examples = examples.sample(n=max_examples, random_state=seed)

    columns = ["query_norm", "cat_l1", "cat_l2", "cat_l3", "cat_leaf"]
    return examples[columns].reset_index(drop=True)


@dataclass
class ConstantCategoryModel:
    """Return a fixed category distribution when training has one class."""

    label: str

    def __post_init__(self) -> None:
        self.classes_ = np.asarray([self.label], dtype=object)

    def predict_proba(self, matrix: Any) -> np.ndarray:
        return np.ones((matrix.shape[0], 1), dtype=np.float32)


class CategorySignal:
    """Fit TF-IDF category classifiers and return each query's top categories."""

    def __init__(
        self,
        levels: tuple[str, ...] = CATEGORY_LEVELS,
        topk: int = CATEGORY_TOPK,
        max_features: int = 120_000,
        max_iter: int = 80,
        min_class_count: int = 2,
        seed: int = 42,
    ):
        self.levels = levels
        self.topk = topk
        self.max_features = max_features
        self.max_iter = max_iter
        self.min_class_count = min_class_count
        self.seed = seed
        self.vectorizer: TfidfVectorizer | None = None
        self.models: dict[str, Any] = {}

    def fit(self, examples: pd.DataFrame) -> "CategorySignal":
        if examples.empty:
            raise ValueError("CategorySignal requires at least one training example")

        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2 if len(examples) >= 2 else 1,
            max_features=self.max_features,
            dtype=np.float32,
            sublinear_tf=True,
        )
        x_all = self.vectorizer.fit_transform(examples["query_norm"])
        print(f"category tfidf shape={x_all.shape}")

        for level in self.levels:
            counts = examples[level].value_counts()
            keep_labels = counts.index[counts >= self.min_class_count]
            if len(keep_labels) == 0:
                keep_labels = counts.index[:1]
            keep = examples[level].isin(keep_labels).to_numpy()
            y = examples.loc[keep, level].astype(str).to_numpy()
            x = x_all[keep]
            classes = np.unique(y)

            start = time.time()
            if len(classes) == 1:
                model: Any = ConstantCategoryModel(str(classes[0]))
            else:
                model = SGDClassifier(
                    loss="log_loss",
                    penalty="l2",
                    alpha=2e-5,
                    max_iter=self.max_iter,
                    tol=1e-3,
                    class_weight="balanced",
                    random_state=self.seed,
                    n_jobs=-1,
                )
                model.fit(x, y)
            self.models[level] = model
            elapsed = time.time() - start
            print(
                f"{level}: classes={len(classes):,} examples={len(y):,} seconds={elapsed:.1f}"
            )
        return self

    def predict_terms(
        self, terms: pd.DataFrame, chunk_size: int = 25_000
    ) -> pd.DataFrame:
        if self.vectorizer is None:
            raise RuntimeError("fit CategorySignal before predicting")
        if terms["term_id"].duplicated().any():
            raise ValueError("terms contains duplicate term_id values")

        term_ids = terms["term_id"].astype(str).to_numpy()
        queries = terms["query"].map(normalize_text).to_numpy()
        chunks: list[pd.DataFrame] = []

        for start in tqdm(range(0, len(terms), chunk_size), desc="category topk"):
            end = min(start + chunk_size, len(terms))
            matrix = self.vectorizer.transform(queries[start:end])
            out: dict[str, object] = {"term_id": term_ids[start:end]}

            for level, model in self.models.items():
                proba = model.predict_proba(matrix)
                k = min(self.topk, proba.shape[1])
                if k == proba.shape[1]:
                    indices = np.argsort(-proba, axis=1)[:, :k]
                else:
                    indices = np.argpartition(proba, -k, axis=1)[:, -k:]
                    local_order = np.argsort(
                        -np.take_along_axis(proba, indices, axis=1), axis=1
                    )
                    indices = np.take_along_axis(indices, local_order, axis=1)
                scores = np.take_along_axis(proba, indices, axis=1).astype(np.float32)
                labels = model.classes_[indices]

                for rank in range(k):
                    out[f"{level}_top{rank + 1}"] = labels[:, rank]
                    out[f"{level}_prob{rank + 1}"] = scores[:, rank]

            chunks.append(pd.DataFrame(out))

        if not chunks:
            return pd.DataFrame(columns=["term_id"])
        return pd.concat(chunks, ignore_index=True)


In [ ]:
%%writefile src/ensemble_scores.py
"""Align, rank-normalize, and blend model scores into competition submissions."""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

SCORE_COLUMNS = ("prob", "score")


def score_col(frame: pd.DataFrame, path: str | Path) -> str:
    for column in SCORE_COLUMNS:
        if column in frame.columns:
            return column
    raise ValueError(f"{path} must contain one of {SCORE_COLUMNS}")


def load_score(path: str | Path, name: str, nrows: int | None = None) -> pd.DataFrame:
    frame = pd.read_csv(
        path, dtype={"id": str, "term_id": str}, keep_default_na=False, nrows=nrows
    )
    required = {"id", "term_id"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")

    column = score_col(frame, path)
    out = frame[["id", "term_id", column]].copy()
    out = out.rename(columns={column: name})
    out[name] = pd.to_numeric(out[name], errors="coerce").astype("float32")

    if out["id"].duplicated().any():
        examples = out.loc[out["id"].duplicated(), "id"].head(5).tolist()
        raise ValueError(f"{path} contains duplicate ids, examples={examples}")
    if out[name].isna().any():
        raise ValueError(f"{path} contains null/non-numeric scores")
    return out


def rank_normalize(frame: pd.DataFrame, column: str, scope: str) -> pd.Series:
    if scope == "global":
        return frame[column].rank(method="average", pct=True).astype("float32")
    if scope == "term":
        return (
            frame.groupby("term_id", sort=False)[column]
            .rank(method="average", pct=True)
            .astype("float32")
        )
    raise ValueError(f"Unknown rank scope: {scope}")


def align_scores(
    catboost: pd.DataFrame, lgbm: pd.DataFrame, transformer: pd.DataFrame
) -> pd.DataFrame:
    merged = catboost.merge(
        lgbm[["id", "term_id", "lgbm"]], on="id", how="inner", suffixes=("", "_lgbm")
    )
    if "term_id_lgbm" in merged.columns:
        mismatch = merged["term_id"].ne(merged["term_id_lgbm"])
        if mismatch.any():
            raise ValueError(
                f"CatBoost/LGBM term_id mismatch rows={int(mismatch.sum())}"
            )
        merged = merged.drop(columns=["term_id_lgbm"])

    merged = merged.merge(
        transformer[["id", "term_id", "transformer"]],
        on="id",
        how="inner",
        suffixes=("", "_transformer"),
    )
    if "term_id_transformer" in merged.columns:
        mismatch = merged["term_id"].ne(merged["term_id_transformer"])
        if mismatch.any():
            raise ValueError(
                f"GBDT/transformer term_id mismatch rows={int(mismatch.sum())}"
            )
        merged = merged.drop(columns=["term_id_transformer"])

    expected = len(catboost)
    if len(merged) != expected:
        raise ValueError(
            f"Score files do not align on id: expected={expected:,} merged={len(merged):,}"
        )
    return merged


def normalized_weighted_sum(
    columns: list[pd.Series], weights: list[float]
) -> pd.Series:
    total = float(sum(weights))
    if total <= 0:
        raise ValueError("Blend weights must sum to a positive value")
    output = np.zeros(len(columns[0]), dtype=np.float32)
    for column, weight in zip(columns, weights):
        output += column.to_numpy(dtype=np.float32) * np.float32(weight / total)
    return pd.Series(output, index=columns[0].index, dtype="float32")


def report_correlations(frame: pd.DataFrame) -> None:
    rank_cols = ["catboost_rank", "lgbm_rank", "transformer_rank", "gbdt_rank", "prob"]
    corr = frame[rank_cols].corr(method="pearson")
    print("rank correlation matrix")
    print(corr.to_string(float_format=lambda value: f"{value:.4f}"))


def blend(args: argparse.Namespace) -> None:
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    nrows = args.limit_rows or None

    catboost = load_score(args.catboost, "catboost", nrows=nrows)
    lgbm = load_score(args.lgbm, "lgbm", nrows=nrows)
    transformer = load_score(args.transformer, "transformer", nrows=nrows)
    merged = align_scores(catboost, lgbm, transformer)

    merged["catboost_rank"] = rank_normalize(merged, "catboost", args.rank_scope)
    merged["lgbm_rank"] = rank_normalize(merged, "lgbm", args.rank_scope)
    merged["transformer_rank"] = rank_normalize(merged, "transformer", args.rank_scope)
    merged["gbdt_score"] = normalized_weighted_sum(
        [merged["catboost_rank"], merged["lgbm_rank"]],
        [args.catboost_weight, args.lgbm_weight],
    )
    merged["gbdt_rank"] = rank_normalize(merged, "gbdt_score", args.rank_scope)
    merged["prob"] = normalized_weighted_sum(
        [merged["gbdt_rank"], merged["transformer_rank"]],
        [args.gbdt_weight, args.transformer_weight],
    )

    columns = ["id", "term_id", "prob"]
    if args.include_components:
        columns.extend(["gbdt_score", "catboost_rank", "lgbm_rank", "transformer_rank"])
    merged[columns].to_csv(output_path, index=False)

    print(f"wrote {output_path} rows={len(merged):,}")
    print(
        "weights "
        f"gbdt=({args.catboost_weight:.3f} catboost, {args.lgbm_weight:.3f} lgbm) "
        f"final=({args.gbdt_weight:.3f} gbdt, {args.transformer_weight:.3f} transformer)"
    )
    if args.report_correlations:
        report_correlations(merged)


def rate_suffix(rate: float) -> str:
    return f"r{int(round(rate * 100)):02d}"


def make_prediction(
    scores: pd.Series,
    rate: float,
    term_ids: pd.Series | None = None,
    min_positive_excess_base: int = 0,
) -> np.ndarray:
    if not 0.0 < rate < 1.0:
        raise ValueError(f"rate must be between 0 and 1, got {rate}")
    n_pos = int(round(len(scores) * rate))
    prediction = np.zeros(len(scores), dtype=np.int8)
    if n_pos <= 0:
        return prediction

    if min_positive_excess_base <= 0:
        selected = scores.nlargest(n_pos).index
        prediction[selected] = 1
        return prediction

    if term_ids is None:
        raise ValueError(
            "term_ids are required when min_positive_excess_base is enabled"
        )
    if len(term_ids) != len(scores):
        raise ValueError("term_ids and scores must have the same number of rows")

    # The competition candidate set appears to contain a base retrieval slate of
    # 100 items per term, unioned with known positives that retrieval missed.  If
    # that inference is correct, a term with N candidates has at least N - 100
    # positives.  Protect the highest-scoring rows needed to satisfy that lower
    # bound, then spend the remaining global positive budget normally.
    frame = pd.DataFrame(
        {
            "term_id": term_ids.astype(str).to_numpy(),
            "score": scores.to_numpy(dtype=np.float64),
        }
    )
    group_size = frame.groupby("term_id", sort=False)["score"].transform("size")
    minimum = (group_size - min_positive_excess_base).clip(lower=0)
    within_term_rank = frame.groupby("term_id", sort=False)["score"].rank(
        method="first",
        ascending=False,
    )
    mandatory = within_term_rank.le(minimum).to_numpy()
    n_mandatory = int(mandatory.sum())
    if n_mandatory > n_pos:
        minimum_rate = n_mandatory / len(scores)
        raise ValueError(
            "Requested positive rate is too small for the per-term lower bounds: "
            f"mandatory={n_mandatory:,} budget={n_pos:,} minimum_rate={minimum_rate:.6f}"
        )

    prediction[mandatory] = 1
    remaining = n_pos - n_mandatory
    if remaining:
        eligible_positions = np.flatnonzero(~mandatory)
        eligible_scores = frame.loc[~mandatory, "score"].to_numpy()
        if remaining >= len(eligible_positions):
            selected_positions = eligible_positions
        else:
            selected_local = np.argpartition(eligible_scores, -remaining)[-remaining:]
            selected_positions = eligible_positions[selected_local]
        prediction[selected_positions] = 1
    return prediction


def submit(args: argparse.Namespace) -> None:
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    scores = pd.read_csv(
        args.scores,
        dtype={"id": str},
        keep_default_na=False,
        nrows=args.limit_rows or None,
    )
    scores = scores.reset_index(drop=True)
    column = args.score_col or score_col(scores, args.scores)
    scores[column] = pd.to_numeric(scores[column], errors="coerce")
    if scores[column].isna().any():
        raise ValueError(f"{args.scores} contains null/non-numeric values in {column}")

    for rate in args.rates:
        prediction = make_prediction(
            scores[column],
            rate,
            term_ids=scores["term_id"] if "term_id" in scores.columns else None,
            min_positive_excess_base=args.min_positive_excess_base,
        )
        output = pd.DataFrame({"id": scores["id"], "prediction": prediction})
        output_path = output_dir / f"{args.prefix}_{rate_suffix(rate)}.csv"
        output.to_csv(output_path, index=False)
        print(
            f"{output_path}: rows={len(output):,} positives={int(prediction.sum()):,} "
            f"rate={prediction.mean():.5f} "
            f"min_positive_excess_base={args.min_positive_excess_base}"
        )


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Rank-normalized two-level GBDT + transformer blending."
    )
    sub = parser.add_subparsers(dest="cmd", required=True)

    blend_cmd = sub.add_parser("blend")
    blend_cmd.add_argument(
        "--catboost", default="outputs/catboost_category_aware_2026-06-30_scores.csv"
    )
    blend_cmd.add_argument("--lgbm", default="outputs/pu_lgbm_v1_scores.csv")
    blend_cmd.add_argument(
        "--transformer", default="outputs/transformer_biencoder_scores.csv"
    )
    blend_cmd.add_argument("--output", default="outputs/moe_gbdt_rankblend_scores.csv")
    blend_cmd.add_argument("--catboost-weight", type=float, default=0.70)
    blend_cmd.add_argument("--lgbm-weight", type=float, default=0.30)
    blend_cmd.add_argument("--gbdt-weight", type=float, default=0.50)
    blend_cmd.add_argument("--transformer-weight", type=float, default=0.50)
    blend_cmd.add_argument("--rank-scope", choices=["global", "term"], default="global")
    blend_cmd.add_argument("--include-components", action="store_true")
    blend_cmd.add_argument("--report-correlations", action="store_true")
    blend_cmd.add_argument("--limit-rows", type=int, default=0)
    blend_cmd.set_defaults(func=blend)

    submit_cmd = sub.add_parser("submit")
    submit_cmd.add_argument("--scores", default="outputs/moe_gbdt_rankblend_scores.csv")
    submit_cmd.add_argument("--output-dir", default="outputs")
    submit_cmd.add_argument("--prefix", default="submission_moe_gbdt_rankblend")
    submit_cmd.add_argument(
        "--rates", type=float, nargs="+", default=[0.18, 0.20, 0.22, 0.24]
    )
    submit_cmd.add_argument("--score-col")
    submit_cmd.add_argument(
        "--min-positive-excess-base",
        type=int,
        default=0,
        help=(
            "Opt-in per-term constraint: require at least max(candidate_count - BASE, 0) "
            "positives per term before filling the remaining global rate budget. "
            "Use 100 to test the inferred base-100 candidate construction."
        ),
    )
    submit_cmd.add_argument("--limit-rows", type=int, default=0)
    submit_cmd.set_defaults(func=submit)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/grouped_oof_validation.py
"""Run leakage-aware term-grouped validation for tree and transformer models."""

from __future__ import annotations

import argparse
import csv
import gc
import itertools
import json
import time
from pathlib import Path
from typing import Iterable

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold
from tqdm import tqdm

from category_signal import CategorySignal, build_category_examples
from pu_catboost import (
    BASE_FEATURE_NAMES,
    CAT_FEATURES,
    CATEGORY_FEATURE_NAMES,
    build_item_meta_lookup,
    build_term_category_lookup,
    rows_to_frame,
)
from lexical_baseline import load_items, load_terms, normalize_text
from train_term_negatives import item_search_text, topk_sparse_row

SLATE_COLUMNS = [
    "slate_id",
    "term_id",
    "item_id",
    "fold",
    "label",
    "candidate_count",
    "is_retrieved",
    "retrieval_rank",
    "retrieval_score",
]
OPTIONAL_SEMANTIC_FEATURES = ["semantic_cosine", "semantic_rank_pct"]


def load_positive_pairs(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "label" in frame.columns:
        frame = frame.loc[frame["label"].astype(str).ne("0")]
    frame = frame[["term_id", "item_id"]].drop_duplicates().reset_index(drop=True)
    if frame.empty:
        raise ValueError(f"No positive pairs found in {path}")
    return frame


def make_term_folds(positives: pd.DataFrame, n_splits: int, seed: int) -> pd.DataFrame:
    counts = (
        positives.groupby("term_id", sort=True)
        .size()
        .rename("n_positives")
        .reset_index()
    )
    if len(counts) < n_splits:
        raise ValueError(f"Need at least {n_splits} terms, found {len(counts)}")

    # Stratifying quantiles of query breadth distributes both head and tail queries
    # across folds while term_id grouping prevents any query leakage.
    n_bins = max(1, min(10, len(counts) // n_splits))
    bins = pd.qcut(
        counts["n_positives"].rank(method="first"),
        q=n_bins,
        labels=False,
        duplicates="drop",
    ).astype(int)
    counts["fold"] = np.int16(-1)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    dummy = np.zeros(len(counts), dtype=np.int8)
    for fold, (_, valid_index) in enumerate(splitter.split(dummy, bins)):
        counts.loc[valid_index, "fold"] = np.int16(fold)

    if counts["fold"].lt(0).any():
        raise AssertionError("Some terms were not assigned to a fold")
    return counts[["term_id", "fold", "n_positives"]]


def sample_zero_score_items(
    item_count: int,
    selected: set[int],
    needed: int,
    rng: np.random.Generator,
) -> list[int]:
    output: list[int] = []
    max_attempts = max(1_000, needed * 100)
    attempts = 0
    while len(output) < needed and attempts < max_attempts:
        position = int(rng.integers(0, item_count))
        attempts += 1
        if position in selected:
            continue
        selected.add(position)
        output.append(position)
    if len(output) != needed:
        for position in range(item_count):
            if position in selected:
                continue
            selected.add(position)
            output.append(position)
            if len(output) == needed:
                break
    if len(output) != needed:
        raise ValueError(f"Could not fill a {needed}-item zero-score retrieval tail")
    return output


def restrict_debug_catalog(
    items: pd.DataFrame,
    positives: pd.DataFrame,
    limit_items: int,
) -> pd.DataFrame:
    if limit_items <= 0 or len(items) <= limit_items:
        return items
    required = set(positives["item_id"].astype(str))
    keep = items["item_id"].astype(str).isin(required)
    head_positions = np.arange(len(items)) < limit_items
    restricted = (
        items.loc[keep | head_positions]
        .drop_duplicates("item_id")
        .reset_index(drop=True)
    )
    print(
        f"debug catalog restricted from {len(items):,} to {len(restricted):,}; "
        f"retained_positive_items={len(required):,}"
    )
    return restricted


def build_slates(args: argparse.Namespace) -> None:
    started = time.time()
    data_dir = Path(args.data_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    positives = load_positive_pairs(data_dir / "training_pairs.csv")
    terms = pd.read_csv(data_dir / "terms.csv", dtype=str, keep_default_na=False)
    train_terms = terms.loc[
        terms["term_id"].isin(set(positives["term_id"])), ["term_id", "query"]
    ].copy()
    train_terms = train_terms.sort_values("term_id").reset_index(drop=True)
    if args.limit_terms:
        train_terms = train_terms.head(args.limit_terms).copy()
        positives = positives.loc[
            positives["term_id"].isin(set(train_terms["term_id"]))
        ].reset_index(drop=True)

    missing_terms = set(positives["term_id"]) - set(train_terms["term_id"])
    if missing_terms:
        raise KeyError(f"Missing {len(missing_terms)} positive term ids from terms.csv")

    folds = make_term_folds(positives, args.n_splits, args.seed)
    folds_path = output_dir / "term_folds.csv"
    folds.to_csv(folds_path, index=False)
    fold_by_term = folds.set_index("term_id")["fold"].astype(int).to_dict()

    items = pd.read_csv(
        data_dir / "items.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["item_id", "title", "category", "brand", "attributes"],
    )
    items = restrict_debug_catalog(items, positives, args.limit_items)
    if len(items) < args.base_candidates:
        raise ValueError(
            f"Catalog has {len(items)} items, fewer than base_candidates={args.base_candidates}"
        )

    item_ids = items["item_id"].astype(str).to_numpy()
    item_to_position = {item_id: position for position, item_id in enumerate(item_ids)}
    missing_items = set(positives["item_id"]) - set(item_to_position)
    if missing_items:
        raise KeyError(f"Missing {len(missing_items)} positive item ids from items.csv")

    train_terms["query_norm"] = train_terms["query"].map(normalize_text)
    item_text = item_search_text(items)
    corpus = pd.concat([train_terms["query_norm"], item_text], ignore_index=True)
    vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, args.word_ngram_max),
        min_df=args.min_df,
        max_features=args.max_features,
        dtype=np.float32,
        norm="l2",
        sublinear_tf=True,
    )
    vectorizer.fit(corpus)
    joblib.dump(vectorizer, output_dir / "retrieval_vectorizer.joblib")
    term_matrix = vectorizer.transform(train_terms["query_norm"])
    item_matrix_t = vectorizer.transform(item_text).T.tocsr()
    print(
        f"terms={len(train_terms):,} catalog_items={len(items):,} positives={len(positives):,} "
        f"vocab={len(vectorizer.vocabulary_):,}"
    )

    positives_by_term: dict[str, list[int]] = {}
    for term_id, group in positives.groupby("term_id", sort=False):
        positives_by_term[str(term_id)] = [
            item_to_position[item_id] for item_id in group["item_id"].astype(str)
        ]

    slate_path = output_dir / "validation_slates.csv"
    rng = np.random.default_rng(args.seed)
    row_id = 0
    positive_rows = 0
    retrieved_positive_rows = 0
    rows_by_fold = np.zeros(args.n_splits, dtype=np.int64)

    with slate_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(SLATE_COLUMNS)

        for start in tqdm(
            range(0, len(train_terms), args.chunk_size), desc="build validation slates"
        ):
            stop = min(start + args.chunk_size, len(train_terms))
            score_chunk = (term_matrix[start:stop] @ item_matrix_t).tocsr()

            for row_offset, term_id in enumerate(
                train_terms["term_id"].iloc[start:stop].astype(str)
            ):
                indices, scores = topk_sparse_row(
                    score_chunk.getrow(row_offset), args.base_candidates
                )
                retrieved_positions = [
                    int(value) for value in indices[: args.base_candidates]
                ]
                retrieved_scores = [
                    float(value) for value in scores[: args.base_candidates]
                ]
                selected = set(retrieved_positions)
                missing = args.base_candidates - len(retrieved_positions)
                if missing:
                    fill = sample_zero_score_items(len(items), selected, missing, rng)
                    retrieved_positions.extend(fill)
                    retrieved_scores.extend([0.0] * len(fill))

                positive_positions = positives_by_term[term_id]
                positive_set = set(positive_positions)
                appended_positions = sorted(positive_set - set(retrieved_positions))
                candidate_positions = retrieved_positions + appended_positions
                candidate_count = len(candidate_positions)
                fold = int(fold_by_term[term_id])

                retrieval_metadata = {
                    position: (rank, score)
                    for rank, (position, score) in enumerate(
                        zip(retrieved_positions, retrieved_scores),
                        start=1,
                    )
                }
                for position in candidate_positions:
                    label = int(position in positive_set)
                    rank, retrieval_score = retrieval_metadata.get(position, (0, 0.0))
                    is_retrieved = int(rank > 0)
                    writer.writerow(
                        [
                            f"OOF_{row_id:010d}",
                            term_id,
                            item_ids[position],
                            fold,
                            label,
                            candidate_count,
                            is_retrieved,
                            rank,
                            f"{retrieval_score:.8f}",
                        ]
                    )
                    row_id += 1
                    positive_rows += label
                    retrieved_positive_rows += label * is_retrieved
                    rows_by_fold[fold] += 1

    if positive_rows != len(positives):
        raise AssertionError(
            f"Expected {len(positives):,} positive slate rows, wrote {positive_rows:,}"
        )

    summary = {
        "n_splits": args.n_splits,
        "seed": args.seed,
        "base_candidates": args.base_candidates,
        "retrieval_max_features": args.max_features,
        "retrieval_min_df": args.min_df,
        "retrieval_word_ngram_max": args.word_ngram_max,
        "terms": len(train_terms),
        "catalog_items": len(items),
        "positive_pairs": len(positives),
        "slate_rows": row_id,
        "positive_rate": positive_rows / row_id,
        "retrieved_positive_recall": retrieved_positive_rows / positive_rows,
        "rows_by_fold": rows_by_fold.tolist(),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    (output_dir / "slate_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, indent=2))
    print(f"wrote {folds_path}")
    print(f"wrote {slate_path}")


def sample_rows(frame: pd.DataFrame, maximum: int, seed: int) -> pd.DataFrame:
    if maximum <= 0 or len(frame) <= maximum:
        return frame.reset_index(drop=True)
    positive = frame.loc[frame["label"].eq(1)]
    negative = frame.loc[frame["label"].eq(0)]
    positive_share = len(positive) / len(frame)
    n_positive = min(len(positive), max(1, int(round(maximum * positive_share))))
    n_negative = min(len(negative), maximum - n_positive)
    parts = [
        (
            positive.sample(n=n_positive, random_state=seed)
            if len(positive) > n_positive
            else positive
        ),
        (
            negative.sample(n=n_negative, random_state=seed + 1)
            if len(negative) > n_negative
            else negative
        ),
    ]
    return (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1.0, random_state=seed + 2)
        .reset_index(drop=True)
    )


def fit_category_signal_for_fold(
    data_dir: Path,
    output_dir: Path,
    train_term_ids: set[str],
    slate_term_ids: set[str],
    args: argparse.Namespace,
) -> pd.DataFrame:
    terms = pd.read_csv(
        data_dir / "terms.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["term_id", "query"],
    )
    items = pd.read_csv(
        data_dir / "items.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["item_id", "category"],
    )
    positives = load_positive_pairs(data_dir / "training_pairs.csv")
    positives = positives.loc[positives["term_id"].isin(train_term_ids)].reset_index(
        drop=True
    )
    examples = build_category_examples(
        terms,
        items,
        positives,
        max_examples=args.category_max_examples or None,
        seed=args.seed,
    )
    signal = CategorySignal(
        topk=args.category_topk,
        max_features=args.category_max_features,
        max_iter=args.category_max_iter,
        min_class_count=args.category_min_class_count,
        seed=args.seed,
    ).fit(examples)
    prediction_terms = terms.loc[
        terms["term_id"].isin(slate_term_ids), ["term_id", "query"]
    ].copy()
    topk = signal.predict_terms(prediction_terms, chunk_size=args.category_chunk_size)
    topk_path = output_dir / "term_category_topk.csv"
    topk.to_csv(topk_path, index=False)
    joblib.dump(signal, output_dir / "category_signal.joblib")
    print(f"wrote {topk_path}")
    return topk


def train_catboost(
    x_fit: pd.DataFrame,
    y_fit: np.ndarray,
    x_inner_valid: pd.DataFrame | None,
    y_inner_valid: np.ndarray | None,
    x_score: pd.DataFrame,
    output_dir: Path,
    args: argparse.Namespace,
) -> tuple[CatBoostClassifier, np.ndarray]:
    params: dict[str, object] = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "iterations": args.catboost_iterations,
        "learning_rate": args.catboost_learning_rate,
        "depth": args.catboost_depth,
        "l2_leaf_reg": args.catboost_l2_leaf_reg,
        "random_strength": args.catboost_random_strength,
        "bootstrap_type": "Bernoulli",
        "subsample": args.catboost_subsample,
        "auto_class_weights": "Balanced",
        "random_seed": args.seed,
        "thread_count": args.threads,
        "allow_writing_files": False,
        "verbose": args.verbose,
        "task_type": args.task_type,
        "devices": args.devices,
    }
    if args.task_type == "CPU":
        params["rsm"] = args.catboost_rsm

    train_pool = Pool(x_fit, y_fit, cat_features=CAT_FEATURES)
    score_pool = Pool(x_score, cat_features=CAT_FEATURES)
    model = CatBoostClassifier(**params)
    if x_inner_valid is not None and y_inner_valid is not None:
        inner_valid_pool = Pool(x_inner_valid, y_inner_valid, cat_features=CAT_FEATURES)
        model.fit(
            train_pool,
            eval_set=inner_valid_pool,
            use_best_model=True,
            early_stopping_rounds=args.early_stopping_rounds,
        )
    else:
        model.fit(train_pool)
    probability = model.predict_proba(score_pool)[:, 1].astype(np.float32)
    model.save_model(str(output_dir / "catboost.cbm"))
    return model, probability


def train_lgbm(
    x_fit: pd.DataFrame,
    y_fit: np.ndarray,
    x_inner_valid: pd.DataFrame | None,
    y_inner_valid: np.ndarray | None,
    x_score: pd.DataFrame,
    output_dir: Path,
    args: argparse.Namespace,
) -> tuple[lgb.LGBMClassifier, np.ndarray]:
    numeric_features = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES
    numeric_features.extend(
        feature for feature in OPTIONAL_SEMANTIC_FEATURES if feature in x_fit.columns
    )
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=args.lgbm_estimators,
        learning_rate=args.lgbm_learning_rate,
        num_leaves=args.lgbm_num_leaves,
        min_child_samples=args.lgbm_min_child_samples,
        subsample=args.lgbm_subsample,
        colsample_bytree=args.lgbm_colsample_bytree,
        reg_lambda=args.lgbm_reg_lambda,
        class_weight="balanced",
        random_state=args.seed,
        n_jobs=args.threads,
    )
    callbacks: list[object] = [lgb.log_evaluation(period=args.verbose)]
    fit_kwargs: dict[str, object] = {"callbacks": callbacks}
    if x_inner_valid is not None and y_inner_valid is not None:
        if args.early_stopping_rounds > 0:
            callbacks.append(
                lgb.early_stopping(args.early_stopping_rounds, verbose=True)
            )
        fit_kwargs["eval_set"] = [(x_inner_valid[numeric_features], y_inner_valid)]
        fit_kwargs["eval_metric"] = "binary_logloss"
    model.fit(x_fit[numeric_features], y_fit, **fit_kwargs)
    probability = model.predict_proba(x_score[numeric_features])[:, 1].astype(
        np.float32
    )
    joblib.dump(
        {"model": model, "features": numeric_features}, output_dir / "lgbm.joblib"
    )
    return model, probability


def inner_group_split(
    rows: pd.DataFrame,
    valid_size: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    if valid_size <= 0.0:
        return np.arange(len(rows), dtype=np.int64), np.empty(0, dtype=np.int64)
    if not 0.0 < valid_size < 1.0:
        raise ValueError(f"inner-valid-size must be in [0, 1), got {valid_size}")
    splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
    fit_index, inner_valid_index = next(
        splitter.split(rows, rows["label"], groups=rows["term_id"])
    )
    if set(rows.iloc[fit_index]["term_id"]) & set(
        rows.iloc[inner_valid_index]["term_id"]
    ):
        raise AssertionError("Term leakage across inner early-stopping split")
    return fit_index, inner_valid_index


def run_fold(args: argparse.Namespace, fold: int | None = None) -> Path:
    started = time.time()
    selected_fold = int(args.fold if fold is None else fold)
    data_dir = Path(args.data_dir)
    root_output_dir = Path(args.output_dir)
    output_dir = root_output_dir / f"fold_{selected_fold}"
    output_dir.mkdir(parents=True, exist_ok=True)

    slates = pd.read_csv(
        args.slates or root_output_dir / "validation_slates.csv",
        dtype={"slate_id": str, "term_id": str, "item_id": str},
        keep_default_na=False,
    )
    required = set(SLATE_COLUMNS)
    missing = required - set(slates.columns)
    if missing:
        raise ValueError(f"Slate file is missing columns: {sorted(missing)}")
    for column in [
        "fold",
        "label",
        "candidate_count",
        "is_retrieved",
        "retrieval_rank",
    ]:
        slates[column] = pd.to_numeric(slates[column], errors="raise").astype(np.int32)
    slates["retrieval_score"] = pd.to_numeric(
        slates["retrieval_score"], errors="raise"
    ).astype(np.float32)

    available_folds = sorted(slates["fold"].unique().tolist())
    if selected_fold not in available_folds:
        raise ValueError(
            f"fold={selected_fold} not present; available={available_folds}"
        )
    train_rows = slates.loc[slates["fold"].ne(selected_fold)].copy()
    valid_rows = slates.loc[slates["fold"].eq(selected_fold)].copy()
    train_rows = sample_rows(
        train_rows, args.max_train_rows, args.seed + selected_fold * 10
    )
    valid_rows = sample_rows(
        valid_rows, args.max_valid_rows, args.seed + selected_fold * 10 + 1
    )
    if train_rows.empty or valid_rows.empty:
        raise ValueError("Fold split produced empty training or validation rows")
    print(
        f"fold={selected_fold} train_rows={len(train_rows):,} valid_rows={len(valid_rows):,} "
        f"train_terms={train_rows['term_id'].nunique():,} "
        f"valid_terms={valid_rows['term_id'].nunique():,}"
    )

    train_term_ids = set(train_rows["term_id"].astype(str))
    slate_term_ids = set(slates["term_id"].astype(str))
    topk = fit_category_signal_for_fold(
        data_dir,
        output_dir,
        train_term_ids,
        slate_term_ids,
        args,
    )
    term_category_lookup = build_term_category_lookup(topk)

    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")
    item_meta_lookup = build_item_meta_lookup(data_dir / "items.csv")
    x_train = rows_to_frame(
        train_rows,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        f"fold {selected_fold} train features",
    )
    x_valid = rows_to_frame(
        valid_rows,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        f"fold {selected_fold} valid features",
    )
    semantic_features: list[str] = []
    if args.semantic_features:
        semantic_features = [
            feature
            for feature in OPTIONAL_SEMANTIC_FEATURES
            if feature in train_rows.columns and feature in valid_rows.columns
        ]
        for feature in semantic_features:
            x_train[feature] = pd.to_numeric(
                train_rows[feature], errors="raise"
            ).to_numpy(dtype=np.float32)
            x_valid[feature] = pd.to_numeric(
                valid_rows[feature], errors="raise"
            ).to_numpy(dtype=np.float32)
        if not semantic_features:
            print("semantic features requested, but the slate contains none")
    y_train = train_rows["label"].to_numpy(dtype=np.int8)
    y_valid = valid_rows["label"].to_numpy(dtype=np.int8)
    fit_index, inner_valid_index = inner_group_split(
        train_rows,
        args.inner_valid_size,
        args.seed + selected_fold * 100,
    )
    x_fit = x_train.iloc[fit_index]
    y_fit = y_train[fit_index]
    if len(inner_valid_index):
        x_inner_valid: pd.DataFrame | None = x_train.iloc[inner_valid_index]
        y_inner_valid: np.ndarray | None = y_train[inner_valid_index]
    else:
        x_inner_valid = None
        y_inner_valid = None
    print(
        f"fold={selected_fold} model_fit_rows={len(fit_index):,} "
        f"inner_early_stop_rows={len(inner_valid_index):,} outer_score_rows={len(valid_rows):,}"
    )

    score_output = valid_rows[SLATE_COLUMNS].copy().reset_index(drop=True)
    score_output["lexical_score"] = x_valid["lexical_score"].to_numpy(dtype=np.float32)
    for feature in semantic_features:
        score_output[feature] = x_valid[feature].to_numpy(dtype=np.float32)
    models = set(args.models)
    if "catboost" in models:
        _, catboost_probability = train_catboost(
            x_fit,
            y_fit,
            x_inner_valid,
            y_inner_valid,
            x_valid,
            output_dir,
            args,
        )
        score_output["catboost_prob"] = catboost_probability
    if "lgbm" in models:
        _, lgbm_probability = train_lgbm(
            x_fit,
            y_fit,
            x_inner_valid,
            y_inner_valid,
            x_valid,
            output_dir,
            args,
        )
        score_output["lgbm_prob"] = lgbm_probability

    score_path = output_dir / "oof_scores.csv"
    score_output.to_csv(score_path, index=False)
    metadata = {
        "fold": selected_fold,
        "models": sorted(models),
        "train_rows": len(train_rows),
        "valid_rows": len(valid_rows),
        "train_terms": len(train_term_ids),
        "valid_terms": valid_rows["term_id"].nunique(),
        "model_fit_rows": len(fit_index),
        "inner_early_stop_rows": len(inner_valid_index),
        "semantic_features": semantic_features,
        "train_positive_rate": float(y_train.mean()),
        "valid_positive_rate": float(y_valid.mean()),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    (output_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    print(json.dumps(metadata, indent=2))
    print(f"wrote {score_path}")

    del x_train, x_valid, terms, items, item_meta_lookup
    gc.collect()
    return score_path


def combine_fold_scores(
    output_dir: Path, n_splits: int, output: Path | None = None
) -> Path:
    paths = [output_dir / f"fold_{fold}" / "oof_scores.csv" for fold in range(n_splits)]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing fold score files: {missing}")
    frames: list[pd.DataFrame] = []
    for fold, path in enumerate(paths):
        frame = pd.read_csv(
            path, dtype={"slate_id": str, "term_id": str, "item_id": str}
        )
        transformer_path = output_dir / f"fold_{fold}" / "transformer_oof_scores.csv"
        if transformer_path.exists():
            transformer = pd.read_csv(transformer_path, dtype={"slate_id": str})
            required = {"slate_id", "transformer_prob"}
            missing_columns = required - set(transformer.columns)
            if missing_columns:
                raise ValueError(
                    f"{transformer_path} is missing {sorted(missing_columns)}"
                )
            frame = frame.merge(
                transformer[["slate_id", "transformer_prob"]],
                on="slate_id",
                how="left",
                validate="one_to_one",
            )
            if frame["transformer_prob"].isna().any():
                raise ValueError(
                    f"{transformer_path} does not cover every row in fold {fold}"
                )
        frames.append(frame)
    combined = pd.concat(frames, ignore_index=True)
    if combined["slate_id"].duplicated().any():
        examples = (
            combined.loc[combined["slate_id"].duplicated(), "slate_id"].head().tolist()
        )
        raise ValueError(f"Duplicate OOF slate ids, examples={examples}")
    combined = combined.sort_values("slate_id").reset_index(drop=True)
    output_path = output or output_dir / "oof_scores.csv"
    combined.to_csv(output_path, index=False)
    print(f"wrote {output_path} rows={len(combined):,}")
    return output_path


def run_all(args: argparse.Namespace) -> None:
    for fold in range(args.n_splits):
        run_fold(args, fold=fold)
    combine_fold_scores(Path(args.output_dir), args.n_splits)


def simplex_weights(n_components: int, step: float) -> Iterable[tuple[float, ...]]:
    if n_components <= 0:
        raise ValueError("At least one score component is required")
    units = int(round(1.0 / step))
    if units <= 0 or not np.isclose(units * step, 1.0, atol=1e-8):
        raise ValueError(
            "weight-step must divide 1.0 exactly, for example 0.25, 0.20, or 0.10"
        )
    for separators in itertools.combinations(
        range(units + n_components - 1), n_components - 1
    ):
        boundaries = (-1,) + separators + (units + n_components - 1,)
        counts = tuple(
            boundaries[i + 1] - boundaries[i] - 1 for i in range(n_components)
        )
        yield tuple(count / units for count in counts)


def macro_f1_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    positive_denominator = 2 * tp + fp + fn
    negative_denominator = 2 * tn + fp + fn
    positive_f1 = 2 * tp / positive_denominator if positive_denominator else 0.0
    negative_f1 = 2 * tn / negative_denominator if negative_denominator else 0.0
    return 0.5 * (positive_f1 + negative_f1)


def evaluate_rates(
    scores: np.ndarray,
    labels: np.ndarray,
    term_ids: np.ndarray,
    rates: list[float],
    constraint_base: int,
) -> list[tuple[float, float, int]]:
    n_rows = len(scores)
    total_positive = int(labels.sum())
    mandatory = np.zeros(n_rows, dtype=bool)
    if constraint_base > 0:
        frame = pd.DataFrame({"term_id": term_ids, "score": scores})
        group_size = frame.groupby("term_id", sort=False)["score"].transform("size")
        minimum = (group_size - constraint_base).clip(lower=0)
        rank = frame.groupby("term_id", sort=False)["score"].rank(
            method="first", ascending=False
        )
        mandatory = rank.le(minimum).to_numpy()

    mandatory_count = int(mandatory.sum())
    mandatory_positive = int(labels[mandatory].sum())
    eligible_positions = np.flatnonzero(~mandatory)
    order = np.argsort(-scores[eligible_positions], kind="stable")
    ordered_labels = labels[eligible_positions[order]].astype(np.int64)
    cumulative_positive = np.cumsum(ordered_labels)

    output: list[tuple[float, float, int]] = []
    for rate in rates:
        predicted_positive = int(round(n_rows * rate))
        if predicted_positive < mandatory_count or predicted_positive > n_rows:
            continue
        additional = predicted_positive - mandatory_count
        additional_tp = int(cumulative_positive[additional - 1]) if additional else 0
        tp = mandatory_positive + additional_tp
        fp = predicted_positive - tp
        fn = total_positive - tp
        tn = n_rows - tp - fp - fn
        output.append((rate, macro_f1_from_counts(tp, fp, fn, tn), mandatory_count))
    return output


def merge_extra_scores(frame: pd.DataFrame, paths: list[str]) -> pd.DataFrame:
    merged = frame
    for raw_path in paths:
        path = Path(raw_path)
        extra = pd.read_csv(path, dtype={"slate_id": str})
        if "slate_id" not in extra.columns:
            raise ValueError(f"{path} must contain slate_id")
        value_columns = [column for column in extra.columns if column != "slate_id"]
        duplicates = set(value_columns) & set(merged.columns)
        if duplicates:
            raise ValueError(
                f"{path} duplicates existing columns: {sorted(duplicates)}"
            )
        merged = merged.merge(extra, on="slate_id", how="left", validate="one_to_one")
        if merged[value_columns].isna().any().any():
            raise ValueError(f"{path} does not cover every OOF slate_id")
    return merged


def optimize(args: argparse.Namespace) -> None:
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(args.oof_scores, dtype={"slate_id": str, "term_id": str})
    frame = merge_extra_scores(frame, args.extra_scores)
    for column in ["label", "fold"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise").astype(np.int16)

    if args.components:
        components = args.components
    else:
        preferred = [
            "catboost_prob",
            "lgbm_prob",
            "transformer_prob",
            "semantic_cosine",
            "semantic_rank_pct",
            "lexical_score",
        ]
        components = [column for column in preferred if column in frame.columns]
    if not components:
        raise ValueError("No score components were found; pass --components explicitly")
    missing = set(components) - set(frame.columns)
    if missing:
        raise ValueError(f"OOF score file is missing components: {sorted(missing)}")

    rank_columns: list[np.ndarray] = []
    for component in components:
        numeric = pd.to_numeric(frame[component], errors="coerce")
        if numeric.isna().any():
            raise ValueError(
                f"Component {component} contains missing/non-numeric values"
            )
        fold_rank = numeric.groupby(frame["fold"], sort=False).rank(
            method="average", pct=True
        )
        rank_columns.append(fold_rank.to_numpy(dtype=np.float32))

    labels = frame["label"].to_numpy(dtype=np.int8)
    term_ids = frame["term_id"].astype(str).to_numpy()
    rates = sorted(set(float(rate) for rate in args.rates))
    constraints = sorted(set(int(base) for base in args.constraint_bases))
    results: list[dict[str, float | int]] = []

    weight_grid = list(simplex_weights(len(components), args.weight_step))
    print(
        f"optimizing rows={len(frame):,} components={components} "
        f"weight_combinations={len(weight_grid)} rates={len(rates)} constraints={constraints}"
    )
    for weights in tqdm(weight_grid, desc="OOF blend grid"):
        blend = np.zeros(len(frame), dtype=np.float32)
        for values, weight in zip(rank_columns, weights):
            if weight:
                blend += values * np.float32(weight)
        for constraint_base in constraints:
            for rate, score, mandatory_count in evaluate_rates(
                blend,
                labels,
                term_ids,
                rates,
                constraint_base,
            ):
                row: dict[str, float | int] = {
                    "macro_f1": score,
                    "rate": rate,
                    "constraint_base": constraint_base,
                    "mandatory_count": mandatory_count,
                }
                for component, weight in zip(components, weights):
                    row[f"weight_{component}"] = weight
                results.append(row)

    result_frame = (
        pd.DataFrame(results)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
    )
    result_path = output_dir / "oof_optimization.csv"
    result_frame.to_csv(result_path, index=False)
    best = result_frame.iloc[0].to_dict()
    report = {
        "rows": len(frame),
        "positive_rate": float(labels.mean()),
        "components": components,
        "weight_step": args.weight_step,
        "best": best,
    }
    report_path = output_dir / "oof_best.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(result_frame.head(args.show_top).to_string(index=False))
    print(f"wrote {result_path}")
    print(f"wrote {report_path}")


def add_training_arguments(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--output-dir", default="outputs/grouped_oof")
    parser.add_argument("--slates")
    parser.add_argument(
        "--models",
        nargs="+",
        choices=["catboost", "lgbm"],
        default=["catboost", "lgbm"],
    )
    parser.add_argument(
        "--max-train-rows", type=int, default=0, help="Debug only; 0 uses all rows."
    )
    parser.add_argument(
        "--max-valid-rows", type=int, default=0, help="Debug only; 0 uses all rows."
    )
    parser.add_argument("--category-topk", type=int, default=5)
    parser.add_argument("--category-max-examples", type=int, default=0)
    parser.add_argument("--category-max-features", type=int, default=120_000)
    parser.add_argument("--category-max-iter", type=int, default=80)
    parser.add_argument("--category-min-class-count", type=int, default=2)
    parser.add_argument("--category-chunk-size", type=int, default=25_000)
    parser.add_argument("--task-type", choices=["CPU", "GPU"], default="CPU")
    parser.add_argument("--devices", default="0")
    parser.add_argument("--threads", type=int, default=-1)
    parser.add_argument("--verbose", type=int, default=100)
    parser.add_argument("--early-stopping-rounds", type=int, default=100)
    parser.add_argument(
        "--inner-valid-size",
        type=float,
        default=0.10,
        help=(
            "Fraction of outer-training terms reserved for early stopping; "
            "outer fold labels are never used."
        ),
    )
    parser.add_argument("--catboost-iterations", type=int, default=2_000)
    parser.add_argument("--catboost-learning-rate", type=float, default=0.035)
    parser.add_argument("--catboost-depth", type=int, default=7)
    parser.add_argument("--catboost-l2-leaf-reg", type=float, default=8.0)
    parser.add_argument("--catboost-random-strength", type=float, default=0.8)
    parser.add_argument("--catboost-subsample", type=float, default=0.85)
    parser.add_argument("--catboost-rsm", type=float, default=0.95)
    parser.add_argument("--lgbm-estimators", type=int, default=4_000)
    parser.add_argument("--lgbm-learning-rate", type=float, default=0.03)
    parser.add_argument("--lgbm-num-leaves", type=int, default=128)
    parser.add_argument("--lgbm-min-child-samples", type=int, default=80)
    parser.add_argument("--lgbm-subsample", type=float, default=0.9)
    parser.add_argument("--lgbm-colsample-bytree", type=float, default=0.9)
    parser.add_argument("--lgbm-reg-lambda", type=float, default=2.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--semantic-features",
        action=argparse.BooleanOptionalAction,
        default=True,
        help=(
            "Use semantic_cosine/semantic_rank_pct when those optional columns "
            "exist in the slate."
        ),
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Leakage-safe term-grouped candidate-slate OOF validation."
    )
    subparsers = parser.add_subparsers(dest="command", required=True)

    slate_parser = subparsers.add_parser("build-slates")
    slate_parser.add_argument("--data-dir", default="data")
    slate_parser.add_argument("--output-dir", default="outputs/grouped_oof")
    slate_parser.add_argument("--n-splits", type=int, default=5)
    slate_parser.add_argument("--base-candidates", type=int, default=100)
    slate_parser.add_argument("--chunk-size", type=int, default=128)
    slate_parser.add_argument("--max-features", type=int, default=250_000)
    slate_parser.add_argument("--min-df", type=int, default=2)
    slate_parser.add_argument("--word-ngram-max", type=int, default=2)
    slate_parser.add_argument("--seed", type=int, default=42)
    slate_parser.add_argument("--limit-terms", type=int, default=0, help="Debug only.")
    slate_parser.add_argument("--limit-items", type=int, default=0, help="Debug only.")
    slate_parser.set_defaults(func=build_slates)

    fold_parser = subparsers.add_parser("run-fold")
    add_training_arguments(fold_parser)
    fold_parser.add_argument("--fold", type=int, required=True)
    fold_parser.set_defaults(func=run_fold)

    all_parser = subparsers.add_parser("run-all")
    add_training_arguments(all_parser)
    all_parser.add_argument("--n-splits", type=int, default=5)
    all_parser.set_defaults(func=run_all)

    combine_parser = subparsers.add_parser("combine")
    combine_parser.add_argument("--output-dir", default="outputs/grouped_oof")
    combine_parser.add_argument("--n-splits", type=int, default=5)
    combine_parser.add_argument("--output")
    combine_parser.set_defaults(
        func=lambda args: combine_fold_scores(
            Path(args.output_dir),
            args.n_splits,
            Path(args.output) if args.output else None,
        )
    )

    optimize_parser = subparsers.add_parser("optimize")
    optimize_parser.add_argument(
        "--oof-scores", default="outputs/grouped_oof/oof_scores.csv"
    )
    optimize_parser.add_argument("--output-dir", default="outputs/grouped_oof")
    optimize_parser.add_argument("--extra-scores", nargs="*", default=[])
    optimize_parser.add_argument("--components", nargs="*")
    optimize_parser.add_argument("--weight-step", type=float, default=0.25)
    optimize_parser.add_argument(
        "--rates",
        nargs="+",
        type=float,
        default=[round(value, 3) for value in np.arange(0.15, 0.351, 0.005)],
    )
    optimize_parser.add_argument(
        "--constraint-bases", nargs="+", type=int, default=[0, 100]
    )
    optimize_parser.add_argument("--show-top", type=int, default=20)
    optimize_parser.set_defaults(func=optimize)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/transformer_biencoder.py
"""Train a transformer pair classifier and score candidate term-item pairs."""

from __future__ import annotations

import argparse
import csv
import gc
import json
import math
import re
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm import tqdm

DEFAULT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
ITEM_COLUMNS = [
    "item_id",
    "title",
    "category",
    "brand",
    "gender",
    "age_group",
    "attributes",
]
SPACE_RE = re.compile(r"\s+")


def require_transformer_stack():
    try:
        import torch
        from torch import nn
        from torch.nn import functional as F
        from torch.utils.data import DataLoader, Dataset
        from transformers import (
            AutoModel,
            AutoTokenizer,
            get_linear_schedule_with_warmup,
        )
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "Transformer bi-encoder training/scoring requires torch and transformers. "
            "Install PyTorch in this environment or run this script on Kaggle GPU."
        ) from exc

    return (
        torch,
        nn,
        F,
        DataLoader,
        Dataset,
        AutoModel,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )


def clean_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return SPACE_RE.sub(" ", str(value).strip())


def build_item_text(row: Any) -> str:
    parts = [
        clean_text(getattr(row, "title", "")),
        clean_text(getattr(row, "brand", "")),
        clean_text(getattr(row, "category", "")),
        clean_text(getattr(row, "gender", "")),
        clean_text(getattr(row, "age_group", "")),
        clean_text(getattr(row, "attributes", "")),
    ]
    return " | ".join(part for part in parts if part)


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def load_terms_texts(
    data_dir: Path, term_ids: set[str] | None = None
) -> dict[str, str]:
    terms = pd.read_csv(data_dir / "terms.csv", dtype=str, keep_default_na=False)
    if term_ids is not None:
        terms = terms.loc[terms["term_id"].isin(term_ids)]
    return {
        str(row.term_id): clean_text(row.query) for row in terms.itertuples(index=False)
    }


def load_item_texts_for_ids(
    data_dir: Path, item_ids: set[str], chunksize: int
) -> dict[str, str]:
    texts: dict[str, str] = {}
    reader = pd.read_csv(
        data_dir / "items.csv",
        usecols=ITEM_COLUMNS,
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    )
    for chunk in tqdm(reader, desc="load item texts"):
        matched = chunk.loc[chunk["item_id"].isin(item_ids)]
        for row in matched.itertuples(index=False):
            texts[str(row.item_id)] = build_item_text(row)
        if len(texts) == len(item_ids):
            break
    return texts


def load_first_item_ids(data_dir: Path, n_items: int) -> set[str]:
    if n_items <= 0:
        return set()
    items = pd.read_csv(
        data_dir / "items.csv",
        usecols=["item_id"],
        dtype=str,
        keep_default_na=False,
        nrows=n_items,
    )
    return set(items["item_id"].astype(str))


def sample_labeled_pairs(frame: pd.DataFrame, n_rows: int, seed: int) -> pd.DataFrame:
    if n_rows <= 0 or len(frame) <= n_rows:
        return frame

    pos = frame.loc[frame["label"].eq(1)]
    neg = frame.loc[frame["label"].eq(0)]
    n_pos = max(1, min(len(pos), round(n_rows * len(pos) / len(frame))))
    n_neg = max(1, min(len(neg), n_rows - n_pos))
    sampled = [
        pos.sample(n=n_pos, random_state=seed) if len(pos) > n_pos else pos,
        neg.sample(n=n_neg, random_state=seed + 1) if len(neg) > n_neg else neg,
    ]
    return (
        pd.concat(sampled, ignore_index=True)
        .sample(frac=1.0, random_state=seed + 2)
        .reset_index(drop=True)
    )


def load_training_frame(args: argparse.Namespace) -> pd.DataFrame:
    data_dir = Path(args.data_dir)
    positives = pd.read_csv(
        data_dir / "training_pairs.csv", usecols=["term_id", "item_id"], dtype=str
    )
    positives = positives.drop_duplicates(["term_id", "item_id"]).reset_index(drop=True)
    positives["label"] = np.int8(1)

    negatives = pd.read_csv(
        args.negatives, usecols=["term_id", "item_id"], dtype=str, keep_default_na=False
    )
    negatives = negatives.drop_duplicates(["term_id", "item_id"]).reset_index(drop=True)
    keep = ~pair_key(negatives).isin(set(pair_key(positives)))
    dropped = int((~keep).sum())
    negatives = negatives.loc[keep].reset_index(drop=True)
    negatives["label"] = np.int8(0)

    if args.limit_terms:
        term_ids = set(
            positives["term_id"].drop_duplicates().head(args.limit_terms).astype(str)
        )
        positives = positives.loc[positives["term_id"].isin(term_ids)]
        negatives = negatives.loc[negatives["term_id"].isin(term_ids)]

    if args.limit_items:
        item_ids = load_first_item_ids(data_dir, args.limit_items)
        positives = positives.loc[positives["item_id"].isin(item_ids)]
        negatives = negatives.loc[negatives["item_id"].isin(item_ids)]

    frame = pd.concat([positives, negatives], ignore_index=True)
    frame = sample_labeled_pairs(frame, args.limit_pairs, args.seed)
    frame = frame.sample(frac=1.0, random_state=args.seed).reset_index(drop=True)

    if frame["label"].nunique() != 2:
        raise ValueError(
            "Training frame must contain both positive and negative rows after limits are applied"
        )

    print(f"dropped_negative_positive_overlap={dropped:,}")
    print(
        f"training_rows={len(frame):,} positives={int(frame['label'].sum()):,} "
        f"positive_rate={frame['label'].mean():.4f}"
    )
    return frame


def load_slate_training_split(
    args: argparse.Namespace,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if args.fold < 0:
        raise ValueError("--fold must be provided when --slates is used")

    slates = pd.read_csv(args.slates, dtype=str, keep_default_na=False)
    required = {"term_id", "item_id", "label", "fold"}
    missing = required - set(slates.columns)
    if missing:
        raise ValueError(f"{args.slates} is missing columns: {sorted(missing)}")
    if "slate_id" not in slates.columns:
        slates["slate_id"] = [f"SLATE_{index:010d}" for index in range(len(slates))]

    slates["fold"] = pd.to_numeric(slates["fold"], errors="raise").astype(np.int16)
    slates["label"] = pd.to_numeric(slates["label"], errors="raise").astype(np.int8)
    if not set(slates["label"].unique()).issubset({0, 1}):
        raise ValueError(f"{args.slates} label values must be binary")
    available_folds = sorted(slates["fold"].unique().tolist())
    if args.fold not in available_folds:
        raise ValueError(
            f"fold={args.fold} not present in {args.slates}; available={available_folds}"
        )

    if args.limit_terms:
        term_ids = set(
            slates["term_id"].drop_duplicates().head(args.limit_terms).astype(str)
        )
        slates = slates.loc[slates["term_id"].isin(term_ids)]
    if args.limit_items:
        item_ids = load_first_item_ids(Path(args.data_dir), args.limit_items)
        slates = slates.loc[slates["item_id"].isin(item_ids)]

    train_frame = slates.loc[slates["fold"].ne(args.fold)].copy()
    valid_frame = slates.loc[slates["fold"].eq(args.fold)].copy()
    train_frame = sample_labeled_pairs(train_frame, args.limit_pairs, args.seed)
    valid_frame = sample_labeled_pairs(valid_frame, args.max_valid_pairs, args.seed + 1)
    train_frame = train_frame.sample(frac=1.0, random_state=args.seed + 2).reset_index(
        drop=True
    )
    valid_frame = valid_frame.reset_index(drop=True)

    if train_frame.empty or valid_frame.empty:
        raise ValueError(
            "Slate fold split produced an empty training or validation frame"
        )
    if train_frame["label"].nunique() != 2 or valid_frame["label"].nunique() != 2:
        raise ValueError(
            "Slate training and validation frames must both contain labels 0 and 1"
        )
    overlap = set(train_frame["term_id"].astype(str)) & set(
        valid_frame["term_id"].astype(str)
    )
    if overlap:
        examples = sorted(overlap)[:5]
        raise AssertionError(f"Term leakage across slate split, examples={examples}")

    print(
        f"slate_fold={args.fold} train_rows={len(train_frame):,} valid_rows={len(valid_frame):,} "
        f"train_terms={train_frame['term_id'].nunique():,} "
        f"valid_terms={valid_frame['term_id'].nunique():,}"
    )
    return train_frame, valid_frame


def split_inner_validation_by_term(
    frame: pd.DataFrame,
    valid_size: float,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not 0.0 < valid_size < 1.0:
        raise ValueError(
            f"--inner-valid-size must be between 0 and 1, got {valid_size}"
        )
    term_ids = frame["term_id"].drop_duplicates().astype(str).to_numpy()
    if len(term_ids) < 2:
        raise ValueError(
            "Need at least two outer-training terms for inner early stopping"
        )
    rng = np.random.default_rng(seed)
    rng.shuffle(term_ids)
    n_valid_terms = min(
        len(term_ids) - 1, max(1, int(round(len(term_ids) * valid_size)))
    )
    valid_term_ids = set(term_ids[:n_valid_terms])
    inner_valid = (
        frame.loc[frame["term_id"].isin(valid_term_ids)].copy().reset_index(drop=True)
    )
    inner_train = (
        frame.loc[~frame["term_id"].isin(valid_term_ids)].copy().reset_index(drop=True)
    )
    overlap = set(inner_train["term_id"].astype(str)) & set(
        inner_valid["term_id"].astype(str)
    )
    if overlap:
        raise AssertionError("Term leakage across transformer inner validation split")
    if inner_train["label"].nunique() != 2 or inner_valid["label"].nunique() != 2:
        raise ValueError(
            "Transformer inner training and validation frames must contain labels 0 and 1"
        )
    return inner_train, inner_valid


def make_pair_head_class(torch, nn):
    class PairHead(nn.Module):
        def __init__(self, embedding_dim: int, hidden_dim: int, dropout: float):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(embedding_dim * 4 + 1, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, query_embedding, item_embedding):
            dot = (query_embedding * item_embedding).sum(dim=1, keepdim=True)
            features = [
                query_embedding,
                item_embedding,
                torch.abs(query_embedding - item_embedding),
                query_embedding * item_embedding,
                dot,
            ]
            return self.net(torch.cat(features, dim=1)).squeeze(1)

    return PairHead


def mean_pool(model_output, attention_mask, F):
    if hasattr(model_output, "last_hidden_state"):
        token_embeddings = model_output.last_hidden_state
    elif isinstance(model_output, dict):
        token_embeddings = model_output["last_hidden_state"]
    else:
        token_embeddings = model_output[0]
    expanded_mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    pooled = (token_embeddings * expanded_mask).sum(dim=1) / expanded_mask.sum(
        dim=1
    ).clamp(min=1e-9)
    return F.normalize(pooled, p=2, dim=1)


def move_batch(batch: dict[str, Any], device: Any) -> dict[str, Any]:
    return {key: value.to(device) for key, value in batch.items()}


def choose_device(torch, requested: str):
    if requested == "auto":
        if torch.cuda.is_available():
            device_ids = list(range(torch.cuda.device_count()))
            return torch.device(f"cuda:{device_ids[0]}"), device_ids
        return torch.device("cpu"), []

    if requested == "cpu":
        return torch.device("cpu"), []

    if requested.startswith("cuda"):
        if not torch.cuda.is_available():
            raise RuntimeError(f"Requested {requested}, but CUDA is not available")
        if requested == "cuda":
            device_ids = list(range(torch.cuda.device_count()))
        elif "," in requested:
            device_ids = [
                int(value) for value in requested.replace("cuda:", "").split(",")
            ]
        else:
            device_ids = [int(requested.split(":")[1])]

        if not device_ids:
            raise ValueError(f"No CUDA devices selected by --device {requested}")
        max_device = torch.cuda.device_count() - 1
        invalid = [
            device_id
            for device_id in device_ids
            if device_id < 0 or device_id > max_device
        ]
        if invalid:
            raise ValueError(
                f"Invalid CUDA device ids {invalid}; available range is 0..{max_device}"
            )
        return torch.device(f"cuda:{device_ids[0]}"), device_ids

    return torch.device(requested), []


def maybe_wrap_data_parallel(torch, module: Any, device_ids: list[int], name: str):
    if len(device_ids) > 1:
        print(f"using DataParallel for {name} on cuda devices {device_ids}")
        return torch.nn.DataParallel(module, device_ids=device_ids)
    return module


def unwrap_parallel(module: Any) -> Any:
    return module.module if hasattr(module, "module") else module


def train(args: argparse.Namespace) -> None:
    torch, nn, F, DataLoader, Dataset, AutoModel, AutoTokenizer, get_scheduler = (
        require_transformer_stack()
    )
    start_time = time.time()
    data_dir = Path(args.data_dir)
    model_dir = Path(args.model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    device, device_ids = choose_device(torch, args.device)
    print(f"device={device} device_ids={device_ids}")

    if args.slates:
        outer_train_frame, score_frame = load_slate_training_split(args)
        train_frame, valid_frame = split_inner_validation_by_term(
            outer_train_frame,
            args.inner_valid_size,
            args.seed + args.fold * 100,
        )
        frame = pd.concat([train_frame, valid_frame, score_frame], ignore_index=True)
        print(
            f"transformer model_fit_rows={len(train_frame):,} "
            f"inner_early_stop_rows={len(valid_frame):,} outer_score_rows={len(score_frame):,}"
        )
    else:
        frame = load_training_frame(args)
        rng = np.random.default_rng(args.seed)
        valid_mask = rng.random(len(frame)) < args.valid_size
        if valid_mask.all() or (~valid_mask).all():
            raise ValueError(
                "Validation split produced an empty train or validation set"
            )
        train_frame = frame.loc[~valid_mask].reset_index(drop=True)
        valid_frame = frame.loc[valid_mask].reset_index(drop=True)
        score_frame = valid_frame

    required_terms = set(frame["term_id"].astype(str))
    required_items = set(frame["item_id"].astype(str))
    term_texts = load_terms_texts(data_dir, required_terms)
    item_texts = load_item_texts_for_ids(data_dir, required_items, args.item_chunksize)
    missing_terms = required_terms - set(term_texts)
    missing_items = required_items - set(item_texts)
    if missing_terms or missing_items:
        raise KeyError(
            f"missing_terms={len(missing_terms)} missing_items={len(missing_items)}"
        )

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    encoder = AutoModel.from_pretrained(args.model_name)
    embedding_dim = int(encoder.config.hidden_size)
    PairHead = make_pair_head_class(torch, nn)
    head = PairHead(
        embedding_dim=embedding_dim, hidden_dim=args.hidden_dim, dropout=args.dropout
    )
    encoder.to(device)
    head.to(device)
    encoder = maybe_wrap_data_parallel(torch, encoder, device_ids, "encoder")
    head = maybe_wrap_data_parallel(torch, head, device_ids, "head")

    class PairDataset(Dataset):
        def __init__(self, rows: pd.DataFrame):
            self.term_ids = rows["term_id"].astype(str).to_numpy()
            self.item_ids = rows["item_id"].astype(str).to_numpy()
            self.labels = rows["label"].astype("float32").to_numpy()

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, index: int):
            term_id = self.term_ids[index]
            item_id = self.item_ids[index]
            return term_texts[term_id], item_texts[item_id], float(self.labels[index])

    def collate(batch):
        query_text, item_text, labels = zip(*batch)
        query_batch = tokenizer(
            list(query_text),
            padding=True,
            truncation=True,
            max_length=args.max_query_length,
            return_tensors="pt",
        )
        item_batch = tokenizer(
            list(item_text),
            padding=True,
            truncation=True,
            max_length=args.max_item_length,
            return_tensors="pt",
        )
        return query_batch, item_batch, torch.tensor(labels, dtype=torch.float32)

    train_loader = DataLoader(
        PairDataset(train_frame),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        collate_fn=collate,
        pin_memory=device.type == "cuda",
    )
    valid_loader = DataLoader(
        PairDataset(valid_frame),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        collate_fn=collate,
        pin_memory=device.type == "cuda",
    )
    score_loader = DataLoader(
        PairDataset(score_frame),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        collate_fn=collate,
        pin_memory=device.type == "cuda",
    )

    no_decay = ["bias", "LayerNorm.weight"]
    grouped_params = [
        {
            "params": [
                p
                for n, p in encoder.named_parameters()
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": args.weight_decay,
            "lr": args.learning_rate,
        },
        {
            "params": [
                p
                for n, p in encoder.named_parameters()
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
            "lr": args.learning_rate,
        },
        {
            "params": head.parameters(),
            "weight_decay": args.weight_decay,
            "lr": args.head_learning_rate,
        },
    ]
    optimizer = torch.optim.AdamW(grouped_params)
    total_steps = math.ceil(len(train_loader) / args.grad_accum_steps) * args.epochs
    scheduler = get_scheduler(
        optimizer,
        num_warmup_steps=int(total_steps * args.warmup_ratio),
        num_training_steps=total_steps,
    )
    neg_count = int((train_frame["label"] == 0).sum())
    pos_count = int((train_frame["label"] == 1).sum())
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([neg_count / max(1, pos_count)], device=device)
    )
    scaler = torch.cuda.amp.GradScaler(enabled=args.fp16 and device.type == "cuda")

    def run_eval() -> tuple[float, np.ndarray]:
        encoder.eval()
        head.eval()
        total_loss = 0.0
        total_rows = 0
        probability_parts: list[np.ndarray] = []
        with torch.no_grad():
            for query_batch, item_batch, labels in tqdm(
                valid_loader, desc="valid", leave=False
            ):
                query_batch = move_batch(query_batch, device)
                item_batch = move_batch(item_batch, device)
                labels = labels.to(device)
                q_emb = mean_pool(
                    encoder(**query_batch), query_batch["attention_mask"], F
                )
                i_emb = mean_pool(
                    encoder(**item_batch), item_batch["attention_mask"], F
                )
                logits = head(q_emb, i_emb)
                loss = criterion(logits, labels)
                total_loss += float(loss.detach().cpu()) * len(labels)
                total_rows += len(labels)
                probability_parts.append(
                    torch.sigmoid(logits).detach().cpu().numpy().astype(np.float32)
                )
        probabilities = (
            np.concatenate(probability_parts)
            if probability_parts
            else np.empty(0, dtype=np.float32)
        )
        return total_loss / max(1, total_rows), probabilities

    def predict_loader(loader: Any, desc: str) -> np.ndarray:
        encoder.eval()
        head.eval()
        probability_parts: list[np.ndarray] = []
        with torch.no_grad():
            for query_batch, item_batch, _ in tqdm(loader, desc=desc, leave=False):
                query_batch = move_batch(query_batch, device)
                item_batch = move_batch(item_batch, device)
                q_emb = mean_pool(
                    encoder(**query_batch), query_batch["attention_mask"], F
                )
                i_emb = mean_pool(
                    encoder(**item_batch), item_batch["attention_mask"], F
                )
                logits = head(q_emb, i_emb)
                probability_parts.append(
                    torch.sigmoid(logits).detach().cpu().numpy().astype(np.float32)
                )
        return (
            np.concatenate(probability_parts)
            if probability_parts
            else np.empty(0, dtype=np.float32)
        )

    best_valid = float("inf")
    best_valid_probability: np.ndarray | None = None
    best_epoch = 0
    for epoch in range(1, args.epochs + 1):
        encoder.train()
        head.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        running_rows = 0

        for step, (query_batch, item_batch, labels) in enumerate(
            tqdm(train_loader, desc=f"epoch {epoch}"), start=1
        ):
            query_batch = move_batch(query_batch, device)
            item_batch = move_batch(item_batch, device)
            labels = labels.to(device)
            with torch.cuda.amp.autocast(enabled=args.fp16 and device.type == "cuda"):
                q_emb = mean_pool(
                    encoder(**query_batch), query_batch["attention_mask"], F
                )
                i_emb = mean_pool(
                    encoder(**item_batch), item_batch["attention_mask"], F
                )
                logits = head(q_emb, i_emb)
                loss = criterion(logits, labels) / args.grad_accum_steps

            scaler.scale(loss).backward()
            running_loss += (
                float(loss.detach().cpu()) * args.grad_accum_steps * len(labels)
            )
            running_rows += len(labels)

            if step % args.grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(encoder.parameters()) + list(head.parameters()),
                    args.max_grad_norm,
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        valid_loss, valid_probability = run_eval()
        train_loss = running_loss / max(1, running_rows)
        print(f"epoch={epoch} train_loss={train_loss:.6f} valid_loss={valid_loss:.6f}")
        if valid_loss < best_valid:
            best_valid = valid_loss
            best_valid_probability = valid_probability.copy()
            best_epoch = epoch
            save_model(model_dir, encoder, tokenizer, head, args, embedding_dim)
            print(f"saved best model to {model_dir}")

    print(f"best_valid_loss={best_valid:.6f}")
    print(f"best_epoch={best_epoch}")
    if args.valid_output:
        if args.slates:
            del encoder, head, optimizer, scheduler, scaler, criterion, grouped_params
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
            torch, F, tokenizer, encoder, head, _ = load_model(model_dir, device)
            encoder = maybe_wrap_data_parallel(torch, encoder, device_ids, "encoder")
            head = maybe_wrap_data_parallel(torch, head, device_ids, "head")
            output_probability = predict_loader(score_loader, "outer OOF score")
        else:
            if best_valid_probability is None:
                raise RuntimeError(
                    "Best validation probabilities were not captured correctly"
                )
            output_probability = best_valid_probability
        if len(output_probability) != len(score_frame):
            raise RuntimeError(
                "Best-checkpoint output probabilities do not align with the score frame"
            )
        valid_output_path = Path(args.valid_output)
        valid_output_path.parent.mkdir(parents=True, exist_ok=True)
        output_columns = ["slate_id"] if "slate_id" in score_frame.columns else []
        if not output_columns:
            output_columns = [
                column
                for column in ["term_id", "item_id", "label"]
                if column in score_frame.columns
            ]
        valid_output = score_frame[output_columns].copy()
        valid_output["transformer_prob"] = output_probability
        valid_output.to_csv(valid_output_path, index=False)
        print(f"wrote {valid_output_path} rows={len(valid_output):,}")
    print(f"elapsed_minutes={(time.time() - start_time) / 60:.1f}")


def save_model(
    model_dir: Path,
    encoder: Any,
    tokenizer: Any,
    head: Any,
    args: argparse.Namespace,
    embedding_dim: int,
) -> None:
    import torch

    encoder_to_save = unwrap_parallel(encoder)
    head_to_save = unwrap_parallel(head)
    encoder_to_save.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    torch.save(head_to_save.state_dict(), model_dir / "pair_head.pt")
    config = {
        "base_model": args.model_name,
        "embedding_dim": embedding_dim,
        "hidden_dim": args.hidden_dim,
        "dropout": args.dropout,
        "max_query_length": args.max_query_length,
        "max_item_length": args.max_item_length,
    }
    (model_dir / "biencoder_config.json").write_text(
        json.dumps(config, indent=2), encoding="utf-8"
    )


def load_model(model_dir: Path, device: Any):
    torch, nn, F, _, _, AutoModel, AutoTokenizer, _ = require_transformer_stack()
    config = json.loads(
        (model_dir / "biencoder_config.json").read_text(encoding="utf-8")
    )
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    encoder = AutoModel.from_pretrained(model_dir)
    PairHead = make_pair_head_class(torch, nn)
    head = PairHead(
        embedding_dim=int(config["embedding_dim"]),
        hidden_dim=int(config["hidden_dim"]),
        dropout=float(config.get("dropout", 0.0)),
    )
    head.load_state_dict(torch.load(model_dir / "pair_head.pt", map_location=device))
    encoder.to(device)
    head.to(device)
    encoder.eval()
    head.eval()
    return torch, F, tokenizer, encoder, head, config


def encode_text_map(
    ids: list[str],
    text_by_id: dict[str, str],
    tokenizer: Any,
    encoder: Any,
    F: Any,
    torch: Any,
    device: Any,
    batch_size: int,
    max_length: int,
    dtype: str,
) -> np.ndarray:
    arrays: list[np.ndarray] = []
    with torch.no_grad():
        for start in tqdm(range(0, len(ids), batch_size), desc="encode texts"):
            batch_ids = ids[start : start + batch_size]
            texts = [text_by_id[item_id] for item_id in batch_ids]
            encoded = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = move_batch(encoded, device)
            embeddings = mean_pool(encoder(**encoded), encoded["attention_mask"], F)
            array = embeddings.detach().cpu().numpy()
            arrays.append(
                array.astype(np.float16 if dtype == "float16" else np.float32)
            )
    return np.vstack(arrays)


def predict(args: argparse.Namespace) -> None:
    model_dir = Path(args.model_dir)
    data_dir = Path(args.data_dir)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch, _, _, _, _, _, _, _ = require_transformer_stack()
    device, device_ids = choose_device(torch, args.device)
    print(f"device={device} device_ids={device_ids}")
    torch, F, tokenizer, encoder, head, config = load_model(model_dir, device)
    encoder = maybe_wrap_data_parallel(torch, encoder, device_ids, "encoder")
    head = maybe_wrap_data_parallel(torch, head, device_ids, "head")

    pairs_for_ids = pd.read_csv(
        data_dir / "submission_pairs.csv",
        usecols=["term_id", "item_id"],
        dtype=str,
        keep_default_na=False,
        nrows=args.limit_pairs or None,
    )
    term_ids = sorted(set(pairs_for_ids["term_id"].astype(str)))
    item_ids = sorted(set(pairs_for_ids["item_id"].astype(str)))
    print(f"unique_terms={len(term_ids):,} unique_items={len(item_ids):,}")

    term_texts = load_terms_texts(data_dir, set(term_ids))
    item_texts = load_item_texts_for_ids(data_dir, set(item_ids), args.item_chunksize)
    missing_terms = set(term_ids) - set(term_texts)
    missing_items = set(item_ids) - set(item_texts)
    if missing_terms or missing_items:
        raise KeyError(
            f"missing_terms={len(missing_terms)} missing_items={len(missing_items)}"
        )

    term_embeddings = encode_text_map(
        term_ids,
        term_texts,
        tokenizer,
        encoder,
        F,
        torch,
        device,
        args.encode_batch_size,
        int(config["max_query_length"]),
        args.embedding_dtype,
    )
    item_embeddings = encode_text_map(
        item_ids,
        item_texts,
        tokenizer,
        encoder,
        F,
        torch,
        device,
        args.encode_batch_size,
        int(config["max_item_length"]),
        args.embedding_dtype,
    )
    term_pos = {term_id: i for i, term_id in enumerate(term_ids)}
    item_pos = {item_id: i for i, item_id in enumerate(item_ids)}
    del term_texts, item_texts, pairs_for_ids
    gc.collect()

    written = 0
    reader = pd.read_csv(
        data_dir / "submission_pairs.csv",
        usecols=["id", "term_id", "item_id"],
        dtype=str,
        keep_default_na=False,
        chunksize=args.pair_chunksize,
        nrows=args.limit_pairs or None,
    )
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "term_id", "prob"])
        with torch.no_grad():
            for chunk in tqdm(reader, desc="score pairs"):
                q_idx = np.fromiter(
                    (term_pos[str(value)] for value in chunk["term_id"]),
                    dtype=np.int64,
                    count=len(chunk),
                )
                i_idx = np.fromiter(
                    (item_pos[str(value)] for value in chunk["item_id"]),
                    dtype=np.int64,
                    count=len(chunk),
                )
                probs: list[np.ndarray] = []
                for start in range(0, len(chunk), args.score_batch_size):
                    stop = min(start + args.score_batch_size, len(chunk))
                    q_emb = torch.from_numpy(term_embeddings[q_idx[start:stop]]).to(
                        device=device, dtype=torch.float32
                    )
                    i_emb = torch.from_numpy(item_embeddings[i_idx[start:stop]]).to(
                        device=device, dtype=torch.float32
                    )
                    logits = head(q_emb, i_emb)
                    probs.append(torch.sigmoid(logits).detach().cpu().numpy())
                prob_values = np.concatenate(probs)
                writer.writerows(
                    (row_id, term_id, f"{prob:.8f}")
                    for row_id, term_id, prob in zip(
                        chunk["id"], chunk["term_id"], prob_values
                    )
                )
                written += len(chunk)

    print(f"wrote {output_path} rows={written:,}")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Train and score a transformer bi-encoder expert."
    )
    sub = parser.add_subparsers(dest="cmd", required=True)

    train_cmd = sub.add_parser("train")
    train_cmd.add_argument("--data-dir", default="data")
    train_cmd.add_argument("--negatives", default="outputs/train_term_negatives.csv")
    train_cmd.add_argument(
        "--slates",
        help="Grouped OOF validation slate CSV. When set, --fold selects held-out terms.",
    )
    train_cmd.add_argument("--fold", type=int, default=-1)
    train_cmd.add_argument(
        "--valid-output", help="Write best-epoch held-out probabilities to this CSV."
    )
    train_cmd.add_argument("--model-dir", default="outputs/transformer_biencoder")
    train_cmd.add_argument("--model-name", default=DEFAULT_MODEL_NAME)
    train_cmd.add_argument("--epochs", type=int, default=2)
    train_cmd.add_argument("--batch-size", type=int, default=64)
    train_cmd.add_argument("--grad-accum-steps", type=int, default=1)
    train_cmd.add_argument("--learning-rate", type=float, default=2e-5)
    train_cmd.add_argument("--head-learning-rate", type=float, default=1e-4)
    train_cmd.add_argument("--weight-decay", type=float, default=0.01)
    train_cmd.add_argument("--warmup-ratio", type=float, default=0.06)
    train_cmd.add_argument("--max-grad-norm", type=float, default=1.0)
    train_cmd.add_argument("--hidden-dim", type=int, default=256)
    train_cmd.add_argument("--dropout", type=float, default=0.1)
    train_cmd.add_argument("--valid-size", type=float, default=0.05)
    train_cmd.add_argument(
        "--inner-valid-size",
        type=float,
        default=0.10,
        help=(
            "With --slates, reserve this fraction of outer-training terms for "
            "checkpoint selection."
        ),
    )
    train_cmd.add_argument("--max-query-length", type=int, default=48)
    train_cmd.add_argument("--max-item-length", type=int, default=192)
    train_cmd.add_argument("--item-chunksize", type=int, default=100_000)
    train_cmd.add_argument("--num-workers", type=int, default=0)
    train_cmd.add_argument(
        "--device", default="auto", help="auto, cpu, cuda, cuda:0, or cuda:0,1"
    )
    train_cmd.add_argument("--fp16", action="store_true")
    train_cmd.add_argument("--seed", type=int, default=42)
    train_cmd.add_argument("--limit-pairs", type=int, default=0)
    train_cmd.add_argument("--max-valid-pairs", type=int, default=0)
    train_cmd.add_argument("--limit-terms", type=int, default=0)
    train_cmd.add_argument("--limit-items", type=int, default=0)
    train_cmd.set_defaults(func=train)

    pred_cmd = sub.add_parser("predict")
    pred_cmd.add_argument("--data-dir", default="data")
    pred_cmd.add_argument("--model-dir", default="outputs/transformer_biencoder")
    pred_cmd.add_argument(
        "--output", default="outputs/transformer_biencoder_scores.csv"
    )
    pred_cmd.add_argument("--encode-batch-size", type=int, default=512)
    pred_cmd.add_argument("--score-batch-size", type=int, default=8192)
    pred_cmd.add_argument("--pair-chunksize", type=int, default=200_000)
    pred_cmd.add_argument("--item-chunksize", type=int, default=100_000)
    pred_cmd.add_argument(
        "--embedding-dtype", choices=["float32", "float16"], default="float16"
    )
    pred_cmd.add_argument(
        "--device", default="auto", help="auto, cpu, cuda, cuda:0, or cuda:0,1"
    )
    pred_cmd.add_argument("--limit-pairs", type=int, default=0)
    pred_cmd.set_defaults(func=predict)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
sys.path.insert(0, str(SRC_DIR))
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_DIR)
env.setdefault("TOKENIZERS_PARALLELISM", "false")


def run_command(arguments):
    command = [sys.executable, *map(str, arguments)]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, check=True, env=env)


run_command([SRC_DIR / "grouped_oof_validation.py", "--help"])
print("embedded source import check passed")

## Stage 1 — build five-fold candidate slates

In [ ]:
slate_path = OUT_DIR / "validation_slates.csv"
should_build = STAGE in {"build", "all"}
if should_build and (not SKIP_EXISTING or not slate_path.exists()):
    run_command(
        [
            SRC_DIR / "grouped_oof_validation.py",
            "build-slates",
            "--data-dir",
            DATA_DIR,
            "--output-dir",
            OUT_DIR,
            "--n-splits",
            N_SPLITS,
            "--base-candidates",
            BASE_CANDIDATES,
            "--max-features",
            250000,
            "--min-df",
            2,
            "--word-ngram-max",
            2,
            "--chunk-size",
            128,
            "--seed",
            SEED,
        ]
    )
elif should_build:
    print(f"skip existing {slate_path}")
else:
    print("build stage disabled")

if STAGE in {"folds", "optimize"} and not slate_path.exists():
    raise FileNotFoundError(
        f"{slate_path} is required. Run STAGE='build' first or set CACHE_ROOT to a saved cache."
    )

In [ ]:
import json
import pandas as pd

if (OUT_DIR / "slate_summary.json").exists():
    print(
        json.dumps(json.loads((OUT_DIR / "slate_summary.json").read_text()), indent=2)
    )
if slate_path.exists():
    slate_preview = pd.read_csv(slate_path, nrows=5)
    display(slate_preview)

## Stage 2 — fold-specific CatBoost and LightGBM

In [ ]:
should_run_folds = STAGE in {"folds", "all"}
models = []
if RUN_CATBOOST:
    models.append("catboost")
if RUN_LGBM:
    models.append("lgbm")

if should_run_folds and models:
    for fold in FOLDS_TO_RUN:
        fold_score_path = OUT_DIR / f"fold_{fold}" / "oof_scores.csv"
        if SKIP_EXISTING and fold_score_path.exists():
            print(f"skip existing {fold_score_path}")
            continue
        arguments = [
            SRC_DIR / "grouped_oof_validation.py",
            "run-fold",
            "--data-dir",
            DATA_DIR,
            "--output-dir",
            OUT_DIR,
            "--slates",
            slate_path,
            "--fold",
            fold,
            "--models",
            *models,
            "--task-type",
            "GPU" if RUN_CATBOOST else "CPU",
            "--devices",
            CATBOOST_DEVICES,
            "--threads",
            -1,
            "--inner-valid-size",
            INNER_VALID_SIZE,
            "--early-stopping-rounds",
            100,
            "--catboost-iterations",
            CATBOOST_ITERATIONS,
            "--lgbm-estimators",
            LGBM_ESTIMATORS,
            "--seed",
            SEED,
        ]
        run_command(arguments)
elif not should_run_folds:
    print("GBDT fold stage disabled")
else:
    print("No GBDT models selected")

## Stage 3 — optional transformer folds

In [ ]:
if should_run_folds and RUN_TRANSFORMER:
    for fold in FOLDS_TO_RUN:
        fold_dir = OUT_DIR / f"fold_{fold}"
        transformer_score_path = fold_dir / "transformer_oof_scores.csv"
        transformer_model_dir = fold_dir / "transformer"
        if SKIP_EXISTING and transformer_score_path.exists():
            print(f"skip existing {transformer_score_path}")
            continue
        run_command(
            [
                SRC_DIR / "transformer_biencoder.py",
                "train",
                "--data-dir",
                DATA_DIR,
                "--slates",
                slate_path,
                "--fold",
                fold,
                "--model-dir",
                transformer_model_dir,
                "--valid-output",
                transformer_score_path,
                "--model-name",
                BASE_MODEL,
                "--epochs",
                TRANSFORMER_EPOCHS,
                "--batch-size",
                TRANSFORMER_BATCH_SIZE,
                "--grad-accum-steps",
                TRANSFORMER_GRAD_ACCUM,
                "--inner-valid-size",
                INNER_VALID_SIZE,
                "--device",
                TRANSFORMER_DEVICE,
                "--fp16",
                "--seed",
                SEED,
            ]
        )
elif not should_run_folds:
    print("transformer fold stage disabled")
else:
    print("RUN_TRANSFORMER=False; skipping transformer folds")

## Combine completed OOF folds

In [ ]:
gbdt_fold_paths = [
    OUT_DIR / f"fold_{fold}" / "oof_scores.csv" for fold in range(N_SPLITS)
]
all_gbdt_folds_ready = all(path.exists() for path in gbdt_fold_paths)
if all_gbdt_folds_ready:
    run_command(
        [
            SRC_DIR / "grouped_oof_validation.py",
            "combine",
            "--output-dir",
            OUT_DIR,
            "--n-splits",
            N_SPLITS,
        ]
    )
else:
    missing = [str(path) for path in gbdt_fold_paths if not path.exists()]
    print("OOF combine waits for all five GBDT folds. Missing:")
    print("\n".join(missing))

## Stage 4 — optimize Macro-F1 blend, rate, and constraint

In [ ]:
combined_oof_path = OUT_DIR / "oof_scores.csv"
should_optimize = STAGE in {"optimize", "all"}
if should_optimize and combined_oof_path.exists():
    transformer_fold_paths = [
        OUT_DIR / f"fold_{fold}" / "transformer_oof_scores.csv"
        for fold in range(N_SPLITS)
    ]
    components = []
    if RUN_CATBOOST:
        components.append("catboost_prob")
    if RUN_LGBM:
        components.append("lgbm_prob")
    if all(path.exists() for path in transformer_fold_paths):
        components.append("transformer_prob")
    else:
        print(
            "not every transformer fold is ready; optimizing available GBDT components only"
        )

    run_command(
        [
            SRC_DIR / "grouped_oof_validation.py",
            "optimize",
            "--oof-scores",
            combined_oof_path,
            "--output-dir",
            OUT_DIR,
            "--components",
            *components,
            "--weight-step",
            WEIGHT_STEP,
            "--constraint-bases",
            0,
            BASE_CANDIDATES,
            "--show-top",
            30,
        ]
    )
elif should_optimize:
    print(f"cannot optimize until {combined_oof_path} exists")
else:
    print("optimization stage disabled")

In [ ]:
if (OUT_DIR / "oof_best.json").exists():
    print((OUT_DIR / "oof_best.json").read_text())
if (OUT_DIR / "oof_optimization.csv").exists():
    display(pd.read_csv(OUT_DIR / "oof_optimization.csv").head(30))

## Package outputs for the next Kaggle session

In [ ]:
if PACKAGE_CACHE:
    archive_base = WORK_DIR / "grouped_oof_cache"
    archive_path = shutil.make_archive(
        str(archive_base),
        "gztar",
        root_dir=OUT_DIR.parent,
        base_dir=OUT_DIR.name,
    )
    print(
        f"wrote {archive_path} ({Path(archive_path).stat().st_size / 1024**3:.2f} GiB)"
    )

print("\nAvailable artifacts:")
for path in sorted(OUT_DIR.glob("**/*")):
    if path.is_file():
        print(f"{path.relative_to(OUT_DIR)}  {path.stat().st_size / 1024**2:.1f} MiB")